# MSCNN-BiLSTM Autoencoder — Unsupervised NIDS
### Two-Stage Architecture (Singh & Jang-Jaccard, 2022)
**Stage 1:** Multi-Scale CNN Autoencoder (spatial feature extraction per-flow)
**Stage 2:** Bidirectional LSTM Autoencoder (temporal pattern learning in latent space)

> **Primary Target:** Generalization to CSE-CIC-IDS-2018 (unseen dataset)
> **Secondary Target:** CIC-IDS-2017 detection performance

## 0. SETUP & CONFIGURATION

In [16]:
# ============================================================
# KONFIGURASI UTAMA — ubah di sini saja, jangan hardcode
# ============================================================
import os, random, json, warnings, hashlib, pickle, gc, time
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# --- Reproducibility / Runtime ---
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# --- Paths ---
DRIVE_ROOT        = "/content/drive/MyDrive"
DATA_ROOT         = f"{DRIVE_ROOT}/nids-data/raw"
CIC2017_DIR       = f"{DATA_ROOT}/CIC-IDS2017"
CSE2018_DIR       = f"{DATA_ROOT}/CSE-CIC-IDS2018"
PROJECT_ROOT      = f"{DRIVE_ROOT}/mscnn-bilstm-ae-28-feb"
PREPROCESSED_DIR  = f"{PROJECT_ROOT}/preprocessed"
CHECKPOINT_DIR    = f"{PROJECT_ROOT}/checkpoints"
RESULTS_DIR       = f"{PROJECT_ROOT}/results"
ARTIFACTS_DIR     = f"{PROJECT_ROOT}/artifacts"

# --- New Runtime Flags ---
AUTO_CACHE_PREPROCESSING = True
CACHE_MODE = "auto"
GPU_ACCEL_MODE = "max_speed"
ENABLE_MIXED_PRECISION = True
ENABLE_XLA = True
ENABLE_TF_DATA_PIPELINE = True
ALLOW_NON_DETERMINISTIC = True
PREFETCH_AUTOTUNE = True

# --- Run Flags (manual defaults; may be overridden by auto-cache) ---
RERUN_CLEANING        = True
RERUN_FEATURE_SEL     = True
RERUN_DOMAIN_SHIFT    = True
RERUN_SCALING         = True
RERUN_STAGE1_TRAIN    = True
RERUN_LATENT_EXTRACT  = True
RERUN_WINDOWING       = True
RERUN_STAGE2_TRAIN    = True

# --- Feature Selection ---
VARIANCE_THRESHOLD    = 1e-5
CORRELATION_THRESHOLD = 0.98
KS_SHIFT_THRESHOLD    = 0.3

# --- Windowing ---
WINDOW_SIZE           = 10
MIN_FLOWS_PER_SESSION = 3

# --- Stage 1: MSCNN-AE ---
S1_LATENT_DIM         = 8
S1_DROPOUT            = 0.3
S1_BATCH_SIZE         = 512
S1_MAX_EPOCHS         = 100
S1_LR                 = 1e-3
S1_ES_PATIENCE        = 10
S1_LR_PATIENCE        = 5

# --- Stage 2: LSTM-AE ---
S2_LATENT_DIM         = 4
S2_DROPOUT            = 0.3
S2_BATCH_SIZE         = 256
S2_MAX_EPOCHS         = 100
S2_LR                 = 1e-3
S2_ES_PATIENCE        = 10
S2_LR_PATIENCE        = 5

# --- Threshold ---
K_SIGMA_VALUES        = [1.5, 2.0, 2.5, 3.0]
PERCENTILE_VALUES     = [95, 97, 99, 99.5]

# --- Evaluation ---
CHUNK_SIZE            = 500_000

# --- Anomaly Score Weighting (will be optimized in Section 11) ---
STAGE1_WEIGHT         = 0.5
STAGE2_WEIGHT         = 0.5


def _exists(path):
    return os.path.exists(path)


def _all_exist(paths):
    return all(_exists(p) for p in paths)


def resolve_preprocessing_plan():
    cleaning_files = [
        f"{PREPROCESSED_DIR}/cic2017_benign_cleaned.parquet",
        f"{PREPROCESSED_DIR}/cic2017_benign_meta.parquet",
        f"{PREPROCESSED_DIR}/cic2017_all_raw.parquet",
        f"{ARTIFACTS_DIR}/median_values.json",
    ]
    feature_files = [
        f"{ARTIFACTS_DIR}/selected_features.json",
    ]
    scaling_files = [
        f"{PREPROCESSED_DIR}/X_train.npy",
        f"{PREPROCESSED_DIR}/X_val.npy",
        f"{PREPROCESSED_DIR}/train_session_ids.npy",
        f"{PREPROCESSED_DIR}/val_session_ids.npy",
        f"{PREPROCESSED_DIR}/train_timestamps.npy",
        f"{PREPROCESSED_DIR}/val_timestamps.npy",
        f"{ARTIFACTS_DIR}/robust_scaler.pkl",
    ]
    latent_files = [
        f"{PREPROCESSED_DIR}/latent_train.npy",
        f"{PREPROCESSED_DIR}/latent_val.npy",
    ]
    window_files = [
        f"{PREPROCESSED_DIR}/windows_train.npz",
        f"{PREPROCESSED_DIR}/windows_val.npz",
        f"{PREPROCESSED_DIR}/flow_idx_train.npy",
        f"{PREPROCESSED_DIR}/flow_idx_val.npy",
    ]

    status = {
        'cleaning_ready': _all_exist(cleaning_files),
        'feature_ready': _all_exist(feature_files),
        'scaling_ready': _all_exist(scaling_files),
        'latent_ready': _all_exist(latent_files),
        'window_ready': _all_exist(window_files),
    }

    plan = {}
    plan['RERUN_CLEANING'] = not status['cleaning_ready']
    plan['RERUN_FEATURE_SEL'] = plan['RERUN_CLEANING'] or (not status['feature_ready'])
    plan['RERUN_DOMAIN_SHIFT'] = plan['RERUN_FEATURE_SEL'] and (not status['feature_ready'])
    plan['RERUN_SCALING'] = plan['RERUN_FEATURE_SEL'] or (not status['scaling_ready'])
    plan['RERUN_LATENT_EXTRACT'] = plan['RERUN_SCALING'] or (not status['latent_ready'])
    plan['RERUN_WINDOWING'] = plan['RERUN_LATENT_EXTRACT'] or (not status['window_ready'])

    print("\n=== Auto Cache Status ===")
    for k, v in status.items():
        print(f"{k:>18}: {v}")

    print("\n=== Effective Preprocessing Flags ===")
    for k in ['RERUN_CLEANING', 'RERUN_FEATURE_SEL', 'RERUN_DOMAIN_SHIFT',
              'RERUN_SCALING', 'RERUN_LATENT_EXTRACT', 'RERUN_WINDOWING']:
        print(f"{k:>18}: {plan[k]}")

    return plan


if AUTO_CACHE_PREPROCESSING and CACHE_MODE == "auto":
    _plan = resolve_preprocessing_plan()
    RERUN_CLEANING = _plan['RERUN_CLEANING']
    RERUN_FEATURE_SEL = _plan['RERUN_FEATURE_SEL']
    RERUN_DOMAIN_SHIFT = _plan['RERUN_DOMAIN_SHIFT']
    RERUN_SCALING = _plan['RERUN_SCALING']
    RERUN_LATENT_EXTRACT = _plan['RERUN_LATENT_EXTRACT']
    RERUN_WINDOWING = _plan['RERUN_WINDOWING']


if GPU_ACCEL_MODE == "max_speed" and ALLOW_NON_DETERMINISTIC:
    os.environ.pop('TF_DETERMINISTIC_OPS', None)
else:
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

import tensorflow as tf
from tensorflow.keras import mixed_precision

tf.random.set_seed(RANDOM_SEED)

gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

if gpus and GPU_ACCEL_MODE == "max_speed":
    if ENABLE_MIXED_PRECISION:
        mixed_precision.set_global_policy('mixed_float16')
    if ENABLE_XLA:
        tf.config.optimizer.set_jit(True)
else:
    mixed_precision.set_global_policy('float32')

policy_name = mixed_precision.global_policy().name
xla_status = bool(tf.config.optimizer.get_jit())
deterministic_status = os.environ.get('TF_DETERMINISTIC_OPS', '0') == '1'

print(f"TensorFlow {tf.__version__}")
print("\n=== Runtime Summary ===")
print(f"GPU count           : {len(gpus)}")
print(f"GPU names           : {[g.name for g in gpus] if gpus else []}")
print(f"Mixed precision     : {policy_name}")
print(f"XLA JIT             : {xla_status}")
print(f"Deterministic ops   : {deterministic_status}")

print("\n=== Configuration Loaded ===")
print(f"Random seed: {RANDOM_SEED}")
print(f"Stage 1 latent dim: {S1_LATENT_DIM}")
print(f"Stage 2 latent dim: {S2_LATENT_DIM}")
print(f"Window size: {WINDOW_SIZE}")
print(f"AUTO_CACHE_PREPROCESSING: {AUTO_CACHE_PREPROCESSING}")
print(f"GPU_ACCEL_MODE: {GPU_ACCEL_MODE}")

TensorFlow 2.19.0
GPU available: []

=== Configuration Loaded ===
Random seed: 42
Stage 1 latent dim: 8
Stage 2 latent dim: 4
Window size: 10


## 1. MOUNT GOOGLE DRIVE & DIRECTORY SETUP

In [17]:
from google.colab import drive
drive.mount('/content/drive')

for d in [PREPROCESSED_DIR, CHECKPOINT_DIR, RESULTS_DIR, ARTIFACTS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive mounted. Directories ready.")
print(f"CIC-2017 : {CIC2017_DIR}")
print(f"CSE-2018 : {CSE2018_DIR}")
print(f"Project  : {PROJECT_ROOT}")

cic_files = [f for f in os.listdir(CIC2017_DIR) if f.endswith('.csv')]
print(f"\nCIC-2017 CSV files ({len(cic_files)}):")
for f in sorted(cic_files):
    print(f"  - {f}")

cse_files = [f for f in os.listdir(CSE2018_DIR) if f.endswith('.csv')]
print(f"\nCSE-2018 CSV files ({len(cse_files)}):")
for f in sorted(cse_files):
    print(f"  - {f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Directories ready.
CIC-2017 : /content/drive/MyDrive/nids-data/raw/CIC-IDS2017
CSE-2018 : /content/drive/MyDrive/nids-data/raw/CSE-CIC-IDS2018
Project  : /content/drive/MyDrive/mscnn-bilstm-ae-28-feb

CIC-2017 CSV files (8):
  - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  - Friday-WorkingHours-Morning.pcap_ISCX.csv
  - Monday-WorkingHours.pcap_ISCX.csv
  - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  - Tuesday-WorkingHours.pcap_ISCX.csv
  - Wednesday-workingHours.pcap_ISCX.csv

CSE-2018 CSV files (3):
  - 02-14-2018.csv
  - 02-15-2018.csv
  - 02-16-2018.csv


## 2. DATA LOADING & CLEANING

Pipeline: `str('Infinity') → np.inf → np.nan → median imputation`

**Key decisions:**
- Session ID and timestamps are extracted BEFORE dropping columns
- Protocol is retained as a numeric feature (already integer-encoded: 6=TCP, 17=UDP)
- Median imputer is fit ONLY on CIC-2017 benign data

In [18]:
# ============================================================
# 2.1 Load CIC-IDS-2017 — All CSVs, filter BENIGN only
# ============================================================
if RERUN_CLEANING:
    print("=== Loading CIC-IDS-2017 ===")

    csv_files = sorted([
        os.path.join(CIC2017_DIR, f)
        for f in os.listdir(CIC2017_DIR) if f.endswith('.csv')
    ])

    dfs = []
    for fpath in csv_files:
        print(f"  Loading {os.path.basename(fpath)}...", end=" ")
        df = pd.read_csv(fpath, low_memory=False, encoding='utf-8')
        df.columns = df.columns.str.strip()
        print(f"{len(df)} rows, {len(df.columns)} cols")
        dfs.append(df)

    df_all = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal CIC-2017 rows: {len(df_all):,}")
    print(f"Label distribution:\n{df_all['Label'].value_counts()}")

    label_col = 'Label'
    df_benign = df_all[df_all[label_col].str.strip().str.upper() == 'BENIGN'].copy()
    print(f"\nBenign flows: {len(df_benign):,} ({len(df_benign)/len(df_all)*100:.1f}%)")

    del dfs
    gc.collect()
else:
    print("[SKIP] Data loading — using cached data")

=== Loading CIC-IDS-2017 ===
  Loading Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv... 225745 rows, 79 cols
  Loading Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv... 286467 rows, 79 cols
  Loading Friday-WorkingHours-Morning.pcap_ISCX.csv... 191033 rows, 79 cols
  Loading Monday-WorkingHours.pcap_ISCX.csv... 529918 rows, 79 cols
  Loading Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv... 288602 rows, 79 cols
  Loading Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv... 170366 rows, 79 cols
  Loading Tuesday-WorkingHours.pcap_ISCX.csv... 445909 rows, 79 cols
  Loading Wednesday-workingHours.pcap_ISCX.csv... 692703 rows, 79 cols

Total CIC-2017 rows: 2,830,743
Label distribution:
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowlo

In [19]:
# ============================================================
# 2.2 Extract metadata columns BEFORE dropping them
# ============================================================
if RERUN_CLEANING:
    # --- Standardize column names ---
    col_map = {}
    for c in df_benign.columns:
        cl = c.lower().strip()
        if 'src' in cl and 'ip' in cl:
            col_map[c] = 'Src_IP'
        elif 'dst' in cl and 'ip' in cl:
            col_map[c] = 'Dst_IP'
        elif 'source' in cl and 'ip' in cl:
            col_map[c] = 'Src_IP'
        elif 'destination' in cl and 'ip' in cl:
            col_map[c] = 'Dst_IP'
        elif cl == 'protocol':
            col_map[c] = 'Protocol'
        elif 'timestamp' in cl:
            col_map[c] = 'Timestamp'
        elif cl == 'label':
            col_map[c] = 'Label'
        elif 'flow' in cl and 'id' in cl:
            col_map[c] = 'Flow_ID'
        elif ('src' in cl or 'source' in cl) and 'port' in cl:
            col_map[c] = 'Src_Port'
        elif ('dst' in cl or 'destination' in cl) and 'port' in cl:
            col_map[c] = 'Dst_Port'

    df_benign_renamed = df_benign.rename(columns=col_map)
    df_all_renamed = df_all.rename(columns=col_map)

    # Construct session_id BEFORE dropping IP/Protocol columns
    def make_session_id(row):
        key = f"{row.get('Src_IP', 'unk')}_{row.get('Dst_IP', 'unk')}_{row.get('Protocol', 'unk')}"
        return hashlib.md5(key.encode()).hexdigest()[:12]

    print("Constructing session IDs...")
    df_benign_renamed['session_id'] = df_benign_renamed.apply(make_session_id, axis=1)

    # Extract timestamps for windowing
    if 'Timestamp' in df_benign_renamed.columns:
        ts_raw = df_benign_renamed['Timestamp'].copy()
        try:
            df_benign_renamed['ts_parsed'] = pd.to_datetime(ts_raw, format='mixed', dayfirst=True)
        except Exception:
            try:
                df_benign_renamed['ts_parsed'] = pd.to_datetime(ts_raw, infer_datetime_format=True)
            except Exception:
                df_benign_renamed['ts_parsed'] = pd.to_numeric(ts_raw, errors='coerce')
        print(f"Timestamps parsed. Sample: {df_benign_renamed['ts_parsed'].iloc[:3].tolist()}")
    else:
        df_benign_renamed['ts_parsed'] = np.arange(len(df_benign_renamed))
        print("[PERINGATAN] No timestamp column found — using row index as proxy")

    # Save metadata for later use
    meta_cols = ['session_id', 'ts_parsed']
    benign_meta = df_benign_renamed[meta_cols].copy()

    print(f"Unique sessions: {df_benign_renamed['session_id'].nunique():,}")
    print(f"Session ID sample: {df_benign_renamed['session_id'].iloc[:3].tolist()}")

Constructing session IDs...
[PERINGATAN] No timestamp column found — using row index as proxy
Unique sessions: 1
Session ID sample: ['cbaae3029779', 'cbaae3029779', 'cbaae3029779']


In [20]:
# ============================================================
# 2.3 Drop non-numeric columns & handle Protocol
# ============================================================
if RERUN_CLEANING:
    drop_patterns = ['Label', 'Flow_ID', 'Timestamp', 'Src_IP', 'Dst_IP',
                     'Src_Port', 'Dst_Port', 'session_id', 'ts_parsed']

    cols_to_drop = [c for c in df_benign_renamed.columns
                    if any(p in c for p in drop_patterns)]

    # [SARAN] Protocol is already numeric (6=TCP, 17=UDP, 1=ICMP).
    # Keeping it as-is since the integer values carry ordinal meaning
    # relevant to network behavior. No encoding needed.
    if 'Protocol' in cols_to_drop:
        cols_to_drop.remove('Protocol')
        print("[SARAN] Keeping 'Protocol' as numeric feature (6=TCP, 17=UDP, 1=ICMP)")

    print(f"Dropping columns: {cols_to_drop}")
    df_numeric = df_benign_renamed.drop(columns=cols_to_drop, errors='ignore')

    # Handle 'Infinity' string values → np.inf → np.nan
    for col in df_numeric.columns:
        if df_numeric[col].dtype == object:
            df_numeric[col] = df_numeric[col].replace(['Infinity', 'infinity', '-Infinity', '-infinity', 'inf', '-inf'], np.nan)
            df_numeric[col] = pd.to_numeric(df_numeric[col], errors='coerce')

    df_numeric = df_numeric.replace([np.inf, -np.inf], np.nan)

    # Force all columns to float
    df_numeric = df_numeric.astype(np.float32)

    nan_pct = (df_numeric.isna().sum() / len(df_numeric) * 100).sort_values(ascending=False)
    print(f"\nTotal features: {len(df_numeric.columns)}")
    print(f"\nNaN percentage (top 10):")
    print(nan_pct.head(10).to_string())

    feature_names_raw = list(df_numeric.columns)
    print(f"\nFeature list ({len(feature_names_raw)}): {feature_names_raw[:10]}...")

Dropping columns: ['Dst_Port', 'Label', 'session_id', 'ts_parsed']

Total features: 77

NaN percentage (top 10):
Flow Bytes/s              0.078175
Flow Packets/s            0.078175
Total Backward Packets    0.000000
Total Fwd Packets         0.000000
Flow Duration             0.000000
Fwd Packet Length Max     0.000000
Fwd Packet Length Min     0.000000
Fwd Packet Length Mean    0.000000
Fwd Packet Length Std     0.000000
Bwd Packet Length Max     0.000000

Feature list (77): ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max']...


In [21]:
# ============================================================
# 2.4 Median Imputation (fit on benign CIC-2017 ONLY)
# ============================================================
if RERUN_CLEANING:
    median_values = df_numeric.median().to_dict()

    with open(f"{ARTIFACTS_DIR}/median_values.json", 'w') as f:
        json.dump({k: float(v) if pd.notna(v) else 0.0 for k, v in median_values.items()}, f, indent=2)
    print(f"Median values saved to {ARTIFACTS_DIR}/median_values.json")

    nan_before = df_numeric.isna().sum().sum()
    df_numeric = df_numeric.fillna(median_values)
    nan_after = df_numeric.isna().sum().sum()
    print(f"NaN imputed: {nan_before:,} → {nan_after:,}")

    assert nan_after == 0, "Still have NaN after imputation!"

    # Save cleaned data
    df_numeric.to_parquet(f"{PREPROCESSED_DIR}/cic2017_benign_cleaned.parquet", index=False)
    benign_meta.to_parquet(f"{PREPROCESSED_DIR}/cic2017_benign_meta.parquet", index=False)
    print(f"\nCleaned data saved: {df_numeric.shape}")
    print(f"Metadata saved: {benign_meta.shape}")

    # Keep full CIC-2017 (all labels) for evaluation in Section 12
    df_all_renamed.to_parquet(f"{PREPROCESSED_DIR}/cic2017_all_raw.parquet", index=False)
    print(f"Full CIC-2017 (all labels) saved for evaluation: {df_all_renamed.shape}")

    del df_all, df_benign, df_all_renamed, df_benign_renamed
    gc.collect()
else:
    required_files = [
        f"{PREPROCESSED_DIR}/cic2017_benign_cleaned.parquet",
        f"{PREPROCESSED_DIR}/cic2017_benign_meta.parquet",
        f"{ARTIFACTS_DIR}/median_values.json",
    ]
    missing = [p for p in required_files if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(
            "Cache incomplete for cleaning: missing " + ", ".join(missing)
        )

    df_numeric = pd.read_parquet(f"{PREPROCESSED_DIR}/cic2017_benign_cleaned.parquet")
    benign_meta = pd.read_parquet(f"{PREPROCESSED_DIR}/cic2017_benign_meta.parquet")
    with open(f"{ARTIFACTS_DIR}/median_values.json") as f:
        median_values = json.load(f)
    print(f"[CACHED] Loaded cleaned data: {df_numeric.shape}")

Median values saved to /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/artifacts/median_values.json
NaN imputed: 3,554 → 0

Cleaned data saved: (2273097, 77)
Metadata saved: (2273097, 2)
Full CIC-2017 (all labels) saved for evaluation: (2830743, 79)


## 3. FEATURE ENGINEERING & SELECTION

All selection criteria computed ONLY from benign CIC-2017 data.
1. Near-zero variance filter
2. High correlation filter

In [22]:
# ============================================================
# 3.1 Near-Zero Variance Filter
# ============================================================
if RERUN_FEATURE_SEL:
    variances = df_numeric.var()
    low_var_mask = variances < VARIANCE_THRESHOLD
    low_var_features = variances[low_var_mask].index.tolist()

    print(f"=== Near-Zero Variance Filter (threshold={VARIANCE_THRESHOLD}) ===")
    print(f"Features before: {len(df_numeric.columns)}")
    print(f"Low-variance features dropped ({len(low_var_features)}):")
    for feat in low_var_features:
        print(f"  - {feat} (var={variances[feat]:.2e})")

    df_selected = df_numeric.drop(columns=low_var_features)
    print(f"Features after NZV filter: {len(df_selected.columns)}")

=== Near-Zero Variance Filter (threshold=1e-05) ===
Features before: 77
Low-variance features dropped (8):
  - Bwd PSH Flags (var=0.00e+00)
  - Bwd URG Flags (var=0.00e+00)
  - Fwd Avg Bytes/Bulk (var=0.00e+00)
  - Fwd Avg Packets/Bulk (var=0.00e+00)
  - Fwd Avg Bulk Rate (var=0.00e+00)
  - Bwd Avg Bytes/Bulk (var=0.00e+00)
  - Bwd Avg Packets/Bulk (var=0.00e+00)
  - Bwd Avg Bulk Rate (var=0.00e+00)
Features after NZV filter: 69


In [23]:
# ============================================================
# 3.2 High Correlation Filter
# ============================================================
if RERUN_FEATURE_SEL:
    corr_matrix = df_selected.corr().abs()
    upper_tri = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    high_corr_pairs = []
    cols_to_drop_corr = set()

    for col in upper_tri.columns:
        correlated = upper_tri.index[upper_tri[col] > CORRELATION_THRESHOLD].tolist()
        for corr_col in correlated:
            pair_corr = corr_matrix.loc[corr_col, col]
            var_col = variances.get(col, 0)
            var_corr = variances.get(corr_col, 0)
            dropped = corr_col if var_corr <= var_col else col
            kept = col if dropped == corr_col else corr_col
            high_corr_pairs.append({
                'feature_1': col, 'feature_2': corr_col,
                'correlation': pair_corr,
                'var_1': var_col, 'var_2': var_corr,
                'dropped': dropped, 'kept': kept
            })
            cols_to_drop_corr.add(dropped)

    print(f"\n=== High Correlation Filter (threshold={CORRELATION_THRESHOLD}) ===")
    print(f"Correlated pairs found: {len(high_corr_pairs)}")
    print(f"Features to drop: {len(cols_to_drop_corr)}")
    for pair in high_corr_pairs[:10]:
        print(f"  |corr|={pair['correlation']:.4f}: {pair['dropped']} (dropped) ↔ {pair['kept']} (kept)")
    if len(high_corr_pairs) > 10:
        print(f"  ... and {len(high_corr_pairs)-10} more pairs")

    df_selected = df_selected.drop(columns=list(cols_to_drop_corr), errors='ignore')
    print(f"Features after correlation filter: {len(df_selected.columns)}")


=== High Correlation Filter (threshold=0.98) ===
Correlated pairs found: 29
Features to drop: 19
  |corr|=0.9991: Total Fwd Packets (dropped) ↔ Total Backward Packets (kept)
  |corr|=0.9970: Total Fwd Packets (dropped) ↔ Total Length of Bwd Packets (kept)
  |corr|=0.9945: Total Backward Packets (dropped) ↔ Total Length of Bwd Packets (kept)
  |corr|=0.9978: Fwd IAT Total (dropped) ↔ Flow Duration (kept)
  |corr|=0.9934: Flow IAT Max (dropped) ↔ Fwd IAT Max (kept)
  |corr|=0.9801: Bwd IAT Total (dropped) ↔ Flow Duration (kept)
  |corr|=0.9857: Fwd Packets/s (dropped) ↔ Flow Packets/s (kept)
  |corr|=1.0000: Fwd PSH Flags (dropped) ↔ SYN Flag Count (kept)
  |corr|=1.0000: Fwd URG Flags (dropped) ↔ CWE Flag Count (kept)
  |corr|=0.9978: RST Flag Count (dropped) ↔ ECE Flag Count (kept)
  ... and 19 more pairs
Features after correlation filter: 50


In [24]:
# ============================================================
# 3.3 Save Feature Selection Results
# ============================================================
if RERUN_FEATURE_SEL:
    selected_features = list(df_selected.columns)

    # Adjust S1_LATENT_DIM based on actual feature count
    n_features = len(selected_features)
    S1_LATENT_DIM = max(4, min(10, n_features // 6))
    print(f"\n[SARAN] Adjusted S1_LATENT_DIM = {S1_LATENT_DIM} "
          f"(n_features={n_features}, compression ratio={n_features/S1_LATENT_DIM:.1f}x)")

    assert S1_LATENT_DIM <= 10, f"Bottleneck too large: {S1_LATENT_DIM} > 10"
    assert n_features / S1_LATENT_DIM >= 4, f"Compression ratio too low: {n_features/S1_LATENT_DIM:.1f}x < 4x"

    with open(f"{ARTIFACTS_DIR}/selected_features.json", 'w') as f:
        json.dump(selected_features, f, indent=2)

    report_rows = []
    for feat in df_numeric.columns:
        status = 'kept' if feat in selected_features else 'dropped_nzv' if feat in low_var_features else 'dropped_corr'
        report_rows.append({'feature': feat, 'variance': float(variances.get(feat, 0)), 'status': status})
    pd.DataFrame(report_rows).to_csv(f"{RESULTS_DIR}/feature_selection_report.csv", index=False)

    print(f"\n=== Feature Selection Summary ===")
    print(f"Original features: {len(df_numeric.columns)}")
    print(f"After NZV filter: {len(df_numeric.columns) - len(low_var_features)}")
    print(f"After corr filter: {len(selected_features)}")
    print(f"Selected features saved to: {ARTIFACTS_DIR}/selected_features.json")
    print(f"Report saved to: {RESULTS_DIR}/feature_selection_report.csv")

    # Apply selection to data
    df_selected = df_numeric[selected_features].copy()
else:
    required_files = [
        f"{ARTIFACTS_DIR}/selected_features.json",
    ]
    missing = [p for p in required_files if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(
            "Cache incomplete for feature selection: missing " + ", ".join(missing)
        )

    with open(f"{ARTIFACTS_DIR}/selected_features.json") as f:
        selected_features = json.load(f)
    n_features = len(selected_features)
    S1_LATENT_DIM = max(4, min(10, n_features // 6))
    df_selected = df_numeric[selected_features].copy()
    print(f"[CACHED] Using {n_features} selected features, S1_LATENT_DIM={S1_LATENT_DIM}")


[SARAN] Adjusted S1_LATENT_DIM = 8 (n_features=50, compression ratio=6.2x)

=== Feature Selection Summary ===
Original features: 77
After NZV filter: 69
After corr filter: 50
Selected features saved to: /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/artifacts/selected_features.json
Report saved to: /content/drive/MyDrive/mscnn-bilstm-ae-28-feb/results/feature_selection_report.csv


## 4. DOMAIN SHIFT ANALYSIS

Diagnostic only — NO transformations fitted on CSE-2018 data.
Uses KS-test to quantify distribution shift per feature between CIC-2017 and CSE-2018 benign flows.

In [25]:
# ============================================================
# 4.1 Load CSE-2018 Benign Sample
# ============================================================
if RERUN_DOMAIN_SHIFT:
    from scipy.stats import ks_2samp

    print("=== Loading CSE-CIC-IDS-2018 (benign sample) ===")

    cse_csv_files = sorted([
        os.path.join(CSE2018_DIR, f)
        for f in os.listdir(CSE2018_DIR) if f.endswith('.csv')
    ])

    cse_benign_chunks = []
    total_benign = 0
    max_benign = 100_000

    for fpath in cse_csv_files:
        if total_benign >= max_benign:
            break
        print(f"  Reading {os.path.basename(fpath)}...", end=" ")
        for chunk in pd.read_csv(fpath, chunksize=CHUNK_SIZE, low_memory=False, encoding='utf-8'):
            chunk.columns = chunk.columns.str.strip()

            label_col_cse = None
            for c in chunk.columns:
                if c.lower().strip() == 'label':
                    label_col_cse = c
                    break
            if label_col_cse is None:
                continue

            benign_mask = chunk[label_col_cse].str.strip().str.lower() == 'benign'
            benign_chunk = chunk[benign_mask]

            if len(benign_chunk) > 0:
                remaining = max_benign - total_benign
                benign_chunk = benign_chunk.head(remaining)
                cse_benign_chunks.append(benign_chunk)
                total_benign += len(benign_chunk)

            if total_benign >= max_benign:
                break
        print(f"(total benign so far: {total_benign:,})")

    cse_benign_raw = pd.concat(cse_benign_chunks, ignore_index=True)
    print(f"\nCSE-2018 benign sample: {len(cse_benign_raw):,} rows")

    # Map CSE column names to match CIC-2017 selected features
    cse_benign_raw.columns = cse_benign_raw.columns.str.strip()

    # Build column name mapping (CSE may have different naming)
    cse_col_map = {}
    cse_cols_lower = {c.lower(): c for c in cse_benign_raw.columns}
    for sf in selected_features:
        sf_lower = sf.lower()
        if sf in cse_benign_raw.columns:
            cse_col_map[sf] = sf
        elif sf_lower in cse_cols_lower:
            cse_col_map[sf] = cse_cols_lower[sf_lower]
        else:
            # Fuzzy match: try removing spaces/underscores
            sf_clean = sf_lower.replace(' ', '').replace('_', '')
            for k, v in cse_cols_lower.items():
                if k.replace(' ', '').replace('_', '') == sf_clean:
                    cse_col_map[sf] = v
                    break

    matched = [sf for sf in selected_features if sf in cse_col_map]
    unmatched = [sf for sf in selected_features if sf not in cse_col_map]

    print(f"Feature mapping: {len(matched)}/{len(selected_features)} matched")
    if unmatched:
        print(f"[PERINGATAN] Unmatched features: {unmatched}")

    # Apply feature selection and imputation
    cse_data = pd.DataFrame()
    for sf in selected_features:
        if sf in cse_col_map:
            col_data = cse_benign_raw[cse_col_map[sf]].copy()
            if col_data.dtype == object:
                col_data = col_data.replace(
                    ['Infinity', 'infinity', '-Infinity', '-infinity', 'inf', '-inf'], np.nan
                )
                col_data = pd.to_numeric(col_data, errors='coerce')
            col_data = col_data.replace([np.inf, -np.inf], np.nan)
            cse_data[sf] = col_data.astype(np.float32)
        else:
            cse_data[sf] = 0.0  # placeholder for unmatched

    # Apply median imputation (from CIC-2017 fit)
    for sf in selected_features:
        if sf in median_values:
            cse_data[sf] = cse_data[sf].fillna(float(median_values[sf]))
        else:
            cse_data[sf] = cse_data[sf].fillna(0.0)

    print(f"CSE-2018 benign preprocessed: {cse_data.shape}")

    del cse_benign_chunks, cse_benign_raw
    gc.collect()

=== Loading CSE-CIC-IDS-2018 (benign sample) ===
  Reading 02-14-2018.csv... (total benign so far: 100,000)

CSE-2018 benign sample: 100,000 rows
Feature mapping: 20/50 matched
[PERINGATAN] Unmatched features: ['Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Bwd Header Length', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'ECE Flag Count', 'Average Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Fwd Header Length.1', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward']
CSE-2018 benign preprocessed: (100000, 50)


In [26]:
# ============================================================
# 4.2 & 4.3 KS-Test & Visualization
# ============================================================
if RERUN_DOMAIN_SHIFT:
    ks_results = []
    for feat in selected_features:
        cic_vals = df_selected[feat].dropna().values
        cse_vals = cse_data[feat].dropna().values
        if len(cic_vals) == 0 or len(cse_vals) == 0:
            ks_results.append({'feature': feat, 'ks_stat': 1.0, 'p_value': 0.0, 'shift': 'high'})
            continue
        stat, pval = ks_2samp(cic_vals[:50000], cse_vals[:50000])
        category = 'high' if stat >= 0.3 else ('medium' if stat >= 0.1 else 'low')
        ks_results.append({'feature': feat, 'ks_stat': round(stat, 4),
                           'p_value': round(pval, 6), 'shift': category})

    ks_df = pd.DataFrame(ks_results).sort_values('ks_stat', ascending=False)

    n_high = (ks_df['shift'] == 'high').sum()
    n_medium = (ks_df['shift'] == 'medium').sum()
    n_low = (ks_df['shift'] == 'low').sum()

    print(f"\n=== Domain Shift Summary ===")
    print(f"High shift (KS ≥ 0.3)  : {n_high} features")
    print(f"Medium shift (0.1-0.3) : {n_medium} features")
    print(f"Low shift (KS < 0.1)   : {n_low} features")
    print(f"\nTop 10 shifted features:")
    print(ks_df.head(10).to_string(index=False))

    # [SARAN] Recommendation on high-shift features
    if n_high > 0:
        print(f"\n[SARAN] {n_high} features have high domain shift (KS ≥ 0.3).")
        print("  These features may hurt generalization to CSE-2018.")
        print("  However, dropping them risks losing discriminative power on CIC-2017.")
        print("  Strategy: KEEP all features but monitor per-feature contribution")
        print("  in evaluation. RobustScaler + clipping helps mitigate scale differences.")
        if n_high > n_features * 0.5:
            print("  [PERINGATAN] >50% features have high shift — generalization will be challenging.")

    # Bar chart visualization
    fig, ax = plt.subplots(figsize=(14, max(5, len(selected_features) * 0.25)))
    colors = {'high': '#e74c3c', 'medium': '#f39c12', 'low': '#27ae60'}
    bars = ax.barh(range(len(ks_df)), ks_df['ks_stat'].values,
                   color=[colors[s] for s in ks_df['shift'].values])
    ax.set_yticks(range(len(ks_df)))
    ax.set_yticklabels(ks_df['feature'].values, fontsize=7)
    ax.set_xlabel('KS Statistic')
    ax.set_title('Domain Shift: CIC-2017 → CSE-2018 (Benign Distributions)')
    ax.axvline(x=0.3, color='red', linestyle='--', alpha=0.7, label='High shift threshold')
    ax.axvline(x=0.1, color='orange', linestyle='--', alpha=0.7, label='Medium shift threshold')
    ax.legend()
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/domain_shift_analysis.png", dpi=150, bbox_inches='tight')
    plt.show()

    ks_df.to_csv(f"{RESULTS_DIR}/domain_shift_analysis.csv", index=False)
    print(f"\nSaved: domain_shift_analysis.csv & .png")

    del cse_data
    gc.collect()


=== Domain Shift Summary ===
High shift (KS ≥ 0.3)  : 30 features
Medium shift (0.1-0.3) : 14 features
Low shift (KS < 0.1)   : 6 features

Top 10 shifted features:
                    feature  ks_stat  p_value shift
             Flow Packets/s   1.0000      0.0  high
       min_seg_size_forward   0.9995      0.0  high
        Fwd Header Length.1   0.9995      0.0  high
          Max Packet Length   0.9486      0.0  high
        Average Packet Size   0.9486      0.0  high
               Flow Bytes/s   0.9486      0.0  high
      Fwd Packet Length Max   0.9434      0.0  high
Total Length of Fwd Packets   0.9434      0.0  high
       Avg Fwd Segment Size   0.9434      0.0  high
          Bwd Header Length   0.9241      0.0  high

[SARAN] 30 features have high domain shift (KS ≥ 0.3).
  These features may hurt generalization to CSE-2018.
  However, dropping them risks losing discriminative power on CIC-2017.
  Strategy: KEEP all features but monitor per-feature contribution
  in evaluati

## 5. SCALING & TRAIN/VAL SPLIT

- Split by `session_id` (not random rows) to prevent session leakage
- RobustScaler fitted ONLY on training split
- Output clipped to [-5, 5]

In [27]:
# ============================================================
# 5.1 Train/Val Split by Session ID
# ============================================================
if RERUN_SCALING:
    from sklearn.preprocessing import RobustScaler

    if len(benign_meta) != len(df_selected):
        raise ValueError(
            f"Length mismatch: benign_meta={len(benign_meta):,} vs df_selected={len(df_selected):,}. "
            "Re-run cleaning/feature-selection so metadata and features stay aligned."
        )

    session_ids = benign_meta['session_id'].fillna('unknown').astype(str).values
    unique_sessions = np.unique(session_ids)

    np.random.seed(RANDOM_SEED)

    if len(unique_sessions) >= 2:
        np.random.shuffle(unique_sessions)
        split_idx = int(len(unique_sessions) * 0.8)
        split_idx = min(max(split_idx, 1), len(unique_sessions) - 1)

        train_sessions = set(unique_sessions[:split_idx])
        val_sessions = set(unique_sessions[split_idx:])

        train_mask = np.array([sid in train_sessions for sid in session_ids], dtype=bool)
        val_mask = ~train_mask
        split_mode = 'session'
    else:
        n_rows = len(df_selected)
        if n_rows < 2:
            raise ValueError(
                f"Not enough rows to split data: n_rows={n_rows}. Need at least 2 rows."
            )

        idx = np.arange(n_rows)
        np.random.shuffle(idx)
        split_idx = int(n_rows * 0.8)
        split_idx = min(max(split_idx, 1), n_rows - 1)

        train_idx = idx[:split_idx]
        val_idx = idx[split_idx:]

        train_mask = np.zeros(n_rows, dtype=bool)
        train_mask[train_idx] = True
        val_mask = ~train_mask
        split_mode = 'row_fallback'

        print("[PERINGATAN] Only one unique session found. Falling back to row-wise split.")

    X_train_raw = df_selected.values[train_mask]
    X_val_raw = df_selected.values[val_mask]

    train_session_ids = session_ids[train_mask]
    val_session_ids = session_ids[val_mask]

    train_timestamps = benign_meta['ts_parsed'].values[train_mask]
    val_timestamps = benign_meta['ts_parsed'].values[val_mask]

    if X_train_raw.shape[0] == 0 or X_val_raw.shape[0] == 0:
        raise ValueError(
            f"Empty split produced (mode={split_mode}): train={X_train_raw.shape[0]}, val={X_val_raw.shape[0]}."
        )

    print(f"=== Train/Val Split ({split_mode}) ===")
    print(f"Unique sessions: {len(unique_sessions):,}")
    print(f"Train flows    : {X_train_raw.shape[0]:,}")
    print(f"Val flows      : {X_val_raw.shape[0]:,}")

=== Train/Val Split (by session) ===
Unique sessions: 1
Train sessions : 0 (0.0%)
Val sessions   : 1 (100.0%)
Train flows    : 0
Val flows      : 2,273,097


In [28]:
# ============================================================
# 5.2 RobustScaler + Clipping
# ============================================================
if RERUN_SCALING:
    if X_train_raw.shape[0] == 0:
        raise ValueError(
            "X_train_raw is empty before scaling. Check split logic in Section 5.1."
        )
    if X_val_raw.shape[0] == 0:
        raise ValueError(
            "X_val_raw is empty before scaling. Check split logic in Section 5.1."
        )

    scaler = RobustScaler()
    X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
    X_val = scaler.transform(X_val_raw).astype(np.float32)

    # Clip to [-5, 5] to limit extreme outliers
    X_train = np.clip(X_train, -5, 5)
    X_val = np.clip(X_val, -5, 5)

    with open(f"{ARTIFACTS_DIR}/robust_scaler.pkl", 'wb') as f:
        pickle.dump(scaler, f)
    print(f"RobustScaler saved to {ARTIFACTS_DIR}/robust_scaler.pkl")

    # Save scaled data
    np.save(f"{PREPROCESSED_DIR}/X_train.npy", X_train)
    np.save(f"{PREPROCESSED_DIR}/X_val.npy", X_val)
    np.save(f"{PREPROCESSED_DIR}/train_session_ids.npy", train_session_ids)
    np.save(f"{PREPROCESSED_DIR}/val_session_ids.npy", val_session_ids)
    np.save(f"{PREPROCESSED_DIR}/train_timestamps.npy", train_timestamps)
    np.save(f"{PREPROCESSED_DIR}/val_timestamps.npy", val_timestamps)

    print(f"\nX_train: {X_train.shape}, range [{X_train.min():.2f}, {X_train.max():.2f}]")
    print(f"X_val  : {X_val.shape}, range [{X_val.min():.2f}, {X_val.max():.2f}]")
    n_features = X_train.shape[1]
    print(f"n_features = {n_features}")
else:
    required_files = [
        f"{PREPROCESSED_DIR}/X_train.npy",
        f"{PREPROCESSED_DIR}/X_val.npy",
        f"{PREPROCESSED_DIR}/train_session_ids.npy",
        f"{PREPROCESSED_DIR}/val_session_ids.npy",
        f"{PREPROCESSED_DIR}/train_timestamps.npy",
        f"{PREPROCESSED_DIR}/val_timestamps.npy",
        f"{ARTIFACTS_DIR}/robust_scaler.pkl",
    ]
    missing = [p for p in required_files if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(
            "Cache incomplete for scaling: missing " + ", ".join(missing)
        )

    X_train = np.load(f"{PREPROCESSED_DIR}/X_train.npy")
    X_val = np.load(f"{PREPROCESSED_DIR}/X_val.npy")
    train_session_ids = np.load(f"{PREPROCESSED_DIR}/train_session_ids.npy", allow_pickle=True)
    val_session_ids = np.load(f"{PREPROCESSED_DIR}/val_session_ids.npy", allow_pickle=True)
    train_timestamps = np.load(f"{PREPROCESSED_DIR}/train_timestamps.npy", allow_pickle=True)
    val_timestamps = np.load(f"{PREPROCESSED_DIR}/val_timestamps.npy", allow_pickle=True)
    with open(f"{ARTIFACTS_DIR}/robust_scaler.pkl", 'rb') as f:
        scaler = pickle.load(f)
    n_features = X_train.shape[1]
    S1_LATENT_DIM = max(4, min(10, n_features // 6))
    print(f"[CACHED] X_train={X_train.shape}, X_val={X_val.shape}, n_features={n_features}")

ValueError: Found array with 0 sample(s) (shape=(0, 50)) while a minimum of 1 is required by RobustScaler.

## 6. STAGE 1 — MSCNN-AE: TRAINING

Multi-Scale CNN Autoencoder for spatial feature extraction.
- Input: per-flow feature vector `(n_features, 1)` — no windowing
- Multi-scale parallel Conv1D branches (kernel 3, 5, 7)
- Bottleneck: `S1_LATENT_DIM` dimensions with LINEAR activation
- Output: LINEAR activation (input is not bounded [0,1])

In [ ]:
# ============================================================
# 6.1 Build MSCNN-AE Architecture
# ============================================================
from tensorflow.keras.layers import (Input, Conv1D, Dense, Flatten, Reshape,
                                      Dropout, BatchNormalization, Concatenate,
                                      Bidirectional, LSTM, RepeatVector,
                                      TimeDistributed, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

def build_mscnn_ae(n_features, latent_dim, dropout_rate):
    """
    Stage 1: Multi-Scale CNN Autoencoder.
    Uses GlobalAveragePooling1D instead of Flatten to reduce parameter count
    and avoid the LARANGAN of Flatten before bottleneck on high-dim intermediate.
    """
    inputs = Input(shape=(n_features, 1), name='input')

    # === ENCODER — Multi-Scale Parallel Branches ===
    b1 = Conv1D(16, kernel_size=3, padding='same', activation='relu')(inputs)
    b1 = BatchNormalization()(b1)

    b2 = Conv1D(16, kernel_size=5, padding='same', activation='relu')(inputs)
    b2 = BatchNormalization()(b2)

    b3 = Conv1D(16, kernel_size=7, padding='same', activation='relu')(inputs)
    b3 = BatchNormalization()(b3)

    x = Concatenate()([b1, b2, b3])  # (batch, n_features, 48)

    x = Conv1D(32, kernel_size=3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout_rate)(x)

    # GlobalAveragePooling1D compresses spatial dimension without Flatten explosion
    # [SARAN] Using GAP instead of Flatten avoids overparameterization
    # (Flatten would create n_features*32 parameters → too many for small bottleneck)
    x = GlobalAveragePooling1D()(x)  # (batch, 32)

    # Bottleneck — LINEAR activation preserves full value range
    bottleneck = Dense(latent_dim, activation='linear', name='bottleneck')(x)

    # === DECODER ===
    x = Dense(32, activation='relu')(bottleneck)
    x = BatchNormalization()(x)
    x = Dense(n_features * 16, activation='relu')(x)
    x = Reshape((n_features, 16))(x)
    x = Conv1D(16, kernel_size=3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    outputs = Conv1D(1, kernel_size=1, activation='linear', name='output')(x)
    outputs = Reshape((n_features,), name='output_reshape')(outputs)

    model = Model(inputs, outputs, name='MSCNN_AE_Stage1')
    encoder = Model(inputs, bottleneck, name='MSCNN_Encoder')
    return model, encoder

model_s1, encoder_s1 = build_mscnn_ae(n_features, S1_LATENT_DIM, S1_DROPOUT)

print("=== MSCNN-AE Architecture ===")
model_s1.summary()
total_params = model_s1.count_params()
compression = n_features / S1_LATENT_DIM
print(f"\nTotal parameters   : {total_params:,}")
print(f"Compression ratio  : {n_features} / {S1_LATENT_DIM} = {compression:.1f}x")
print(f"Bottleneck dim     : {S1_LATENT_DIM}")

assert S1_LATENT_DIM <= 10, f"[LARANGAN] Bottleneck too large: {S1_LATENT_DIM} > 10"
assert compression >= 4, f"[LARANGAN] Compression ratio too low: {compression:.1f}x < 4x"
print("\n✓ Architecture constraints satisfied")

In [ ]:
# ============================================================
# 6.2 Train Stage 1
# ============================================================
if RERUN_STAGE1_TRAIN:
    model_s1.compile(optimizer=Adam(learning_rate=S1_LR), loss='mse')

    callbacks_s1 = [
        EarlyStopping(monitor='val_loss', patience=S1_ES_PATIENCE,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=S1_LR_PATIENCE, min_lr=1e-6, verbose=1),
        ModelCheckpoint(f"{CHECKPOINT_DIR}/stage1_best.keras",
                        monitor='val_loss', save_best_only=True, verbose=1)
    ]

    X_train_s1 = X_train[..., np.newaxis]  # (n, n_features) → (n, n_features, 1)
    X_val_s1 = X_val[..., np.newaxis]

    print(f"Training Stage 1 MSCNN-AE...")
    print(f"  Train: {X_train_s1.shape}, Val: {X_val_s1.shape}")

    if ENABLE_TF_DATA_PIPELINE:
        train_ds_s1 = tf.data.Dataset.from_tensor_slices((X_train_s1, X_train))
        val_ds_s1 = tf.data.Dataset.from_tensor_slices((X_val_s1, X_val))

        shuffle_buffer = min(len(X_train_s1), 200000)
        if shuffle_buffer > 1:
            train_ds_s1 = train_ds_s1.shuffle(
                shuffle_buffer,
                seed=RANDOM_SEED,
                reshuffle_each_iteration=True
            )

        train_ds_s1 = train_ds_s1.batch(S1_BATCH_SIZE, drop_remainder=False)
        val_ds_s1 = val_ds_s1.batch(S1_BATCH_SIZE, drop_remainder=False)

        if PREFETCH_AUTOTUNE:
            train_ds_s1 = train_ds_s1.prefetch(tf.data.AUTOTUNE)
            val_ds_s1 = val_ds_s1.prefetch(tf.data.AUTOTUNE)

        history_s1 = model_s1.fit(
            train_ds_s1,
            validation_data=val_ds_s1,
            epochs=S1_MAX_EPOCHS,
            callbacks=callbacks_s1,
            verbose=1
        )
    else:
        history_s1 = model_s1.fit(
            X_train_s1, X_train,
            validation_data=(X_val_s1, X_val),
            batch_size=S1_BATCH_SIZE,
            epochs=S1_MAX_EPOCHS,
            callbacks=callbacks_s1,
            verbose=1
        )

    # Diagnostics
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history_s1.history['loss'], label='Train Loss')
    ax.plot(history_s1.history['val_loss'], label='Val Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.set_title('Stage 1 MSCNN-AE Training Curve')
    ax.legend()
    ax.set_yscale('log')
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/stage1_training_curve.png", dpi=150, bbox_inches='tight')
    plt.show()

    final_train_loss = history_s1.history['loss'][-1]
    final_val_loss = history_s1.history['val_loss'][-1]
    best_val_loss = min(history_s1.history['val_loss'])
    best_epoch = history_s1.history['val_loss'].index(best_val_loss) + 1

    print(f"\n=== Stage 1 Training Summary ===")
    print(f"Best epoch     : {best_epoch}")
    print(f"Best val_loss  : {best_val_loss:.6f}")
    print(f"Final train    : {final_train_loss:.6f}")
    print(f"Final val      : {final_val_loss:.6f}")

    if final_val_loss > final_train_loss * 2:
        print("[PERINGATAN] val_loss > 2x train_loss — possible overfitting!")

    # Check convergence in first 10 epochs
    early_losses = history_s1.history['loss'][:10]
    if len(early_losses) >= 10 and early_losses[-1] > early_losses[0] * 0.95:
        print("[PERINGATAN] Loss barely decreased in first 10 epochs — check learning rate or architecture")

else:
    ckpt_path = f"{CHECKPOINT_DIR}/stage1_best.keras"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint missing for Stage 1: {ckpt_path}")
    model_s1.load_weights(ckpt_path)
    print("[CACHED] Stage 1 weights loaded from checkpoint")

## 7. STAGE 1 — LATENT VECTOR EXTRACTION

Extract latent representations from Stage 1 encoder for all splits.
These become the input for session-based windowing and Stage 2.

In [ ]:
# ============================================================
# 7. Extract Latent Vectors
# ============================================================
if RERUN_LATENT_EXTRACT:
    def extract_latent(encoder, X, batch_size=S1_BATCH_SIZE):
        X_reshaped = X[..., np.newaxis]
        return encoder.predict(X_reshaped, batch_size=batch_size, verbose=0)

    latent_train = extract_latent(encoder_s1, X_train)
    latent_val = extract_latent(encoder_s1, X_val)

    np.save(f"{PREPROCESSED_DIR}/latent_train.npy", latent_train)
    np.save(f"{PREPROCESSED_DIR}/latent_val.npy", latent_val)

    print(f"=== Latent Vectors Extracted ===")
    print(f"Latent train : {latent_train.shape}")
    print(f"Latent val   : {latent_val.shape}")
    print(f"Latent range : [{latent_train.min():.3f}, {latent_train.max():.3f}]")
    print(f"Latent std   : {latent_train.std(axis=0).mean():.4f} (mean across dims)")

    # Verify latent space is not collapsed
    dim_stds = latent_train.std(axis=0)
    collapsed = (dim_stds < 1e-4).sum()
    if collapsed > 0:
        print(f"[PERINGATAN] {collapsed}/{S1_LATENT_DIM} latent dimensions nearly collapsed (std < 1e-4)")
    else:
        print(f"✓ All {S1_LATENT_DIM} latent dimensions active")
else:
    required_files = [
        f"{PREPROCESSED_DIR}/latent_train.npy",
        f"{PREPROCESSED_DIR}/latent_val.npy",
    ]
    missing = [p for p in required_files if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(
            "Cache incomplete for latent extraction: missing " + ", ".join(missing)
        )

    latent_train = np.load(f"{PREPROCESSED_DIR}/latent_train.npy")
    latent_val = np.load(f"{PREPROCESSED_DIR}/latent_val.npy")
    print(f"[CACHED] Latent: train={latent_train.shape}, val={latent_val.shape}")

## 8. SESSION-BASED WINDOWING (in Latent Space)

**Fundamental change from previous implementation:**
Windowing happens in **latent space**, not raw feature space.
Windows are grouped by session and sorted by timestamp.

In [ ]:
# ============================================================
# 8.1 Session Length Analysis
# ============================================================
if RERUN_WINDOWING:
    session_lengths_train = pd.Series(train_session_ids).value_counts()

    print("=== Session Length Distribution (Train) ===")
    print(session_lengths_train.describe())
    print(f"\nMedian session length      : {session_lengths_train.median():.1f}")
    print(f"% Sessions with 1 flow     : {(session_lengths_train == 1).mean()*100:.1f}%")
    print(f"% Sessions with < 3 flows  : {(session_lengths_train < 3).mean()*100:.1f}%")
    print(f"% Sessions with >= 10 flows: {(session_lengths_train >= 10).mean()*100:.1f}%")

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(session_lengths_train.clip(upper=50).values, bins=50, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Flows per Session')
    ax.set_ylabel('Count')
    ax.set_title('Session Length Distribution (CIC-2017 Benign, Train Split)')
    ax.axvline(x=WINDOW_SIZE, color='red', linestyle='--', label=f'Window size = {WINDOW_SIZE}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/session_length_distribution.png", dpi=150, bbox_inches='tight')
    plt.show()

    # Decide windowing strategy
    pct_single = (session_lengths_train == 1).mean() * 100
    pct_short = (session_lengths_train < MIN_FLOWS_PER_SESSION).mean() * 100
    median_len = session_lengths_train.median()

    USE_WINDOWING = True
    if median_len < 3 or pct_single > 70:
        print(f"\n[PERINGATAN] Median session length = {median_len:.0f}, "
              f"{pct_single:.1f}% sessions have only 1 flow.")
        print("  Session-based windowing will produce degenerate results.")
        print("  Fallback: Using per-flow mode for Stage 2 (window_size=1)")
        USE_WINDOWING = False
        WINDOW_SIZE = 1
    else:
        print(f"\n✓ Session-based windowing viable (median={median_len:.0f}, "
              f"single-flow={pct_single:.1f}%)")

In [ ]:
# ============================================================
# 8.2 Create Windowed Latent Sequences
# ============================================================
if RERUN_WINDOWING:
    def create_session_windows(latent_vectors, session_ids, timestamps,
                                window_size, min_flows):
        """
        Create non-overlapping windows from latent vectors,
        grouped by session and sorted by timestamp.
        """
        if window_size == 1:
            # Per-flow mode: each flow is its own "window"
            windows = latent_vectors[:, np.newaxis, :]  # (n, 1, latent_dim)
            masks = np.zeros((len(latent_vectors), 1), dtype=bool)
            flow_indices = np.arange(len(latent_vectors))
            return windows, masks, flow_indices

        windows, masks, flow_idx_list = [], [], []

        df_temp = pd.DataFrame({
            'session_id': session_ids,
            'timestamp': range(len(latent_vectors)),  # use index as timestamp proxy for sorting
            'latent_idx': range(len(latent_vectors))
        })

        # Use actual timestamps if available and sortable
        if timestamps is not None:
            try:
                ts_numeric = pd.to_numeric(pd.Series(timestamps), errors='coerce')
                if ts_numeric.notna().mean() > 0.5:
                    df_temp['timestamp'] = ts_numeric.values
            except Exception:
                pass  # stick with index-based ordering

        for sid, group in df_temp.groupby('session_id'):
            group = group.sort_values('timestamp')
            indices = group['latent_idx'].values

            if len(indices) < min_flows:
                continue

            for start in range(0, len(indices), window_size):
                chunk = indices[start:start + window_size]
                w = latent_vectors[chunk]
                mask = np.zeros(window_size, dtype=bool)

                if len(chunk) < window_size:
                    pad_len = window_size - len(chunk)
                    w = np.vstack([w, np.zeros((pad_len, latent_vectors.shape[1]))])
                    mask[len(chunk):] = True

                windows.append(w)
                masks.append(mask)
                # Track which flow indices belong to each window
                padded_chunk = np.full(window_size, -1, dtype=np.int64)
                padded_chunk[:len(chunk)] = chunk
                flow_idx_list.append(padded_chunk)

        return np.array(windows), np.array(masks), np.array(flow_idx_list)

    windows_train, masks_train, flow_idx_train = create_session_windows(
        latent_train, train_session_ids, train_timestamps, WINDOW_SIZE, MIN_FLOWS_PER_SESSION
    )
    windows_val, masks_val, flow_idx_val = create_session_windows(
        latent_val, val_session_ids, val_timestamps, WINDOW_SIZE, MIN_FLOWS_PER_SESSION
    )

    print(f"\n=== Windowed Latent Sequences ===")
    print(f"Window size       : {WINDOW_SIZE}")
    print(f"Windows train     : {windows_train.shape}")
    print(f"Windows val       : {windows_val.shape}")
    print(f"% Padded windows  : {masks_train.any(axis=1).mean()*100:.1f}% (train)")

    # Save flow index mapping for error aggregation
    np.save(f"{PREPROCESSED_DIR}/flow_idx_train.npy", flow_idx_train)
    np.save(f"{PREPROCESSED_DIR}/flow_idx_val.npy", flow_idx_val)

## 9. SAVE/LOAD CHECKPOINT

In [ ]:
# ============================================================
# 9. Checkpoint Management
# ============================================================
def save_preprocessed():
    np.savez_compressed(f"{PREPROCESSED_DIR}/windows_train.npz",
                        X=windows_train, mask=masks_train)
    np.savez_compressed(f"{PREPROCESSED_DIR}/windows_val.npz",
                        X=windows_val, mask=masks_val)
    # Save windowing config
    wconfig = {'window_size': WINDOW_SIZE, 'min_flows': MIN_FLOWS_PER_SESSION,
               'use_windowing': USE_WINDOWING, 'n_windows_train': len(windows_train),
               'n_windows_val': len(windows_val)}
    with open(f"{ARTIFACTS_DIR}/windowing_config.json", 'w') as f:
        json.dump(wconfig, f, indent=2)
    print("Preprocessed windows saved to Drive.")

def load_preprocessed():
    global windows_train, masks_train, windows_val, masks_val, flow_idx_train, flow_idx_val
    d = np.load(f"{PREPROCESSED_DIR}/windows_train.npz")
    windows_train, masks_train = d['X'], d['mask']
    d = np.load(f"{PREPROCESSED_DIR}/windows_val.npz")
    windows_val, masks_val = d['X'], d['mask']
    flow_idx_train = np.load(f"{PREPROCESSED_DIR}/flow_idx_train.npy")
    flow_idx_val = np.load(f"{PREPROCESSED_DIR}/flow_idx_val.npy")
    print(f"Loaded: train={windows_train.shape}, val={windows_val.shape}")

if RERUN_WINDOWING:
    save_preprocessed()
else:
    required_files = [
        f"{PREPROCESSED_DIR}/windows_train.npz",
        f"{PREPROCESSED_DIR}/windows_val.npz",
        f"{PREPROCESSED_DIR}/flow_idx_train.npy",
        f"{PREPROCESSED_DIR}/flow_idx_val.npy",
    ]
    missing = [p for p in required_files if not os.path.exists(p)]
    if missing:
        raise FileNotFoundError(
            "Cache incomplete for windowing: missing " + ", ".join(missing)
        )
    load_preprocessed()

## 10. STAGE 2 — LSTM-AE: TRAINING

Bidirectional LSTM Autoencoder for temporal pattern learning in latent space.
- Input: windowed latent vectors `(window_size, S1_LATENT_DIM)`
- Bottleneck: `S2_LATENT_DIM` (further compression of already-compressed vectors)
- Output: reconstructed latent sequence with LINEAR activation

In [ ]:
# ============================================================
# 10.1 Build LSTM-AE Architecture
# ============================================================
def build_lstm_ae(window_size, latent_dim_in, latent_dim_out, dropout_rate):
    """
    Stage 2: Bidirectional LSTM Autoencoder.
    Learns temporal patterns from sequences of Stage 1 latent vectors.
    """
    inputs = Input(shape=(window_size, latent_dim_in), name='seq_input')

    # === ENCODER ===
    x = Bidirectional(LSTM(32, return_sequences=True))(inputs)
    x = Dropout(dropout_rate)(x)
    x = Bidirectional(LSTM(16, return_sequences=False))(x)
    x = Dropout(dropout_rate)(x)

    bottleneck = Dense(latent_dim_out, activation='linear',
                       name='temporal_bottleneck')(x)

    # === DECODER ===
    x = RepeatVector(window_size)(bottleneck)
    x = Bidirectional(LSTM(16, return_sequences=True))(x)
    x = Dropout(dropout_rate)(x)
    x = Bidirectional(LSTM(32, return_sequences=True))(x)
    outputs = TimeDistributed(
        Dense(latent_dim_in, activation='linear')
    )(x)

    model = Model(inputs, outputs, name='BiLSTM_AE_Stage2')
    return model

actual_window = windows_train.shape[1]
actual_latent = windows_train.shape[2]
model_s2 = build_lstm_ae(actual_window, actual_latent, S2_LATENT_DIM, S2_DROPOUT)

print("=== BiLSTM-AE Architecture ===")
model_s2.summary()
s2_params = model_s2.count_params()
s2_compression = (actual_window * actual_latent) / S2_LATENT_DIM
print(f"\nTotal parameters   : {s2_params:,}")
print(f"Temporal compression: ({actual_window}×{actual_latent}) / {S2_LATENT_DIM} = {s2_compression:.1f}x")

In [ ]:
# ============================================================
# 10.2 Train Stage 2
# ============================================================
if RERUN_STAGE2_TRAIN:
    model_s2.compile(optimizer=Adam(learning_rate=S2_LR), loss='mse')

    callbacks_s2 = [
        EarlyStopping(monitor='val_loss', patience=S2_ES_PATIENCE,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=S2_LR_PATIENCE, min_lr=1e-6, verbose=1),
        ModelCheckpoint(f"{CHECKPOINT_DIR}/stage2_best.keras",
                        monitor='val_loss', save_best_only=True, verbose=1)
    ]

    print(f"Training Stage 2 BiLSTM-AE...")
    print(f"  Train: {windows_train.shape}, Val: {windows_val.shape}")

    if ENABLE_TF_DATA_PIPELINE:
        train_ds_s2 = tf.data.Dataset.from_tensor_slices((windows_train, windows_train))
        val_ds_s2 = tf.data.Dataset.from_tensor_slices((windows_val, windows_val))

        shuffle_buffer = min(len(windows_train), 100000)
        if shuffle_buffer > 1:
            train_ds_s2 = train_ds_s2.shuffle(
                shuffle_buffer,
                seed=RANDOM_SEED,
                reshuffle_each_iteration=True
            )

        train_ds_s2 = train_ds_s2.batch(S2_BATCH_SIZE, drop_remainder=False)
        val_ds_s2 = val_ds_s2.batch(S2_BATCH_SIZE, drop_remainder=False)

        if PREFETCH_AUTOTUNE:
            train_ds_s2 = train_ds_s2.prefetch(tf.data.AUTOTUNE)
            val_ds_s2 = val_ds_s2.prefetch(tf.data.AUTOTUNE)

        history_s2 = model_s2.fit(
            train_ds_s2,
            validation_data=val_ds_s2,
            epochs=S2_MAX_EPOCHS,
            callbacks=callbacks_s2,
            verbose=1
        )
    else:
        history_s2 = model_s2.fit(
            windows_train, windows_train,
            validation_data=(windows_val, windows_val),
            batch_size=S2_BATCH_SIZE,
            epochs=S2_MAX_EPOCHS,
            callbacks=callbacks_s2,
            verbose=1
        )

    # Diagnostics
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(history_s2.history['loss'], label='Train Loss')
    axes[0].plot(history_s2.history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('MSE Loss')
    axes[0].set_title('Stage 2 BiLSTM-AE Training Curve')
    axes[0].legend()
    axes[0].set_yscale('log')

    # Compare Stage 1 vs Stage 2 convergence
    try:
        axes[1].plot(history_s1.history['val_loss'], label='Stage 1 Val Loss', alpha=0.7)
    except Exception:
        pass
    axes[1].plot(history_s2.history['val_loss'], label='Stage 2 Val Loss', alpha=0.7)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MSE Loss')
    axes[1].set_title('Stage 1 vs Stage 2 Convergence')
    axes[1].legend()
    axes[1].set_yscale('log')

    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/stage2_training_curve.png", dpi=150, bbox_inches='tight')
    plt.show()

    s2_final_train = history_s2.history['loss'][-1]
    s2_final_val = history_s2.history['val_loss'][-1]
    s2_best_val = min(history_s2.history['val_loss'])
    s2_best_epoch = history_s2.history['val_loss'].index(s2_best_val) + 1

    print(f"\n=== Stage 2 Training Summary ===")
    print(f"Best epoch     : {s2_best_epoch}")
    print(f"Best val_loss  : {s2_best_val:.6f}")
    print(f"Final train    : {s2_final_train:.6f}")
    print(f"Final val      : {s2_final_val:.6f}")

    if s2_final_val > s2_final_train * 2:
        print("[PERINGATAN] val_loss > 2x train_loss — possible overfitting!")
else:
    ckpt_path = f"{CHECKPOINT_DIR}/stage2_best.keras"
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(f"Checkpoint missing for Stage 2: {ckpt_path}")
    model_s2.load_weights(ckpt_path)
    print("[CACHED] Stage 2 weights loaded from checkpoint")

## 11. THRESHOLD DETERMINATION

Thresholds determined ONLY from benign validation data reconstruction errors.
No test set information is used.

**Anomaly score mapping strategy:**
- Stage 1 error: per-flow MSE
- Stage 2 error: per-window MSE → mapped back to each flow in the window
- Combined: weighted sum of Stage 1 and Stage 2 errors per flow

In [ ]:
# ============================================================
# 11.1 Compute Reconstruction Errors (Validation)
# ============================================================
def compute_recon_error(model, X, batch_size=512):
    """MSE per sample across all dimensions."""
    X_pred = model.predict(X, batch_size=batch_size, verbose=0)
    axes = tuple(range(1, X.ndim))
    return np.mean((X - X_pred) ** 2, axis=axes)

# Stage 1: per-flow error on validation
err_s1_val = compute_recon_error(model_s1, X_val[..., np.newaxis])
print(f"Stage 1 val error: mean={err_s1_val.mean():.6f}, std={err_s1_val.std():.6f}")

# Stage 2: per-window error on validation
err_s2_val_windows = compute_recon_error(model_s2, windows_val)
print(f"Stage 2 val error: mean={err_s2_val_windows.mean():.6f}, std={err_s2_val_windows.std():.6f}")

# Map Stage 2 window errors back to per-flow errors
# Each flow gets the error of the window it belongs to
err_s2_val_perflow = np.full(len(X_val), np.nan, dtype=np.float32)
for w_idx in range(len(flow_idx_val)):
    flow_idxs = flow_idx_val[w_idx]
    valid = flow_idxs[flow_idxs >= 0]
    err_s2_val_perflow[valid] = err_s2_val_windows[w_idx]

# Flows not in any window get median Stage 2 error (conservative)
not_windowed = np.isnan(err_s2_val_perflow)
if not_windowed.any():
    err_s2_val_perflow[not_windowed] = np.nanmedian(err_s2_val_perflow)
    print(f"  {not_windowed.sum()} val flows not in windows → assigned median S2 error")

# Normalize both errors to comparable scales before combining
from sklearn.preprocessing import MinMaxScaler
err_s1_scaler = MinMaxScaler()
err_s2_scaler = MinMaxScaler()
err_s1_norm = err_s1_scaler.fit_transform(err_s1_val.reshape(-1, 1)).ravel()
err_s2_norm = err_s2_scaler.fit_transform(err_s2_val_perflow.reshape(-1, 1)).ravel()

# Save scalers for evaluation
with open(f"{ARTIFACTS_DIR}/error_scalers.pkl", 'wb') as f:
    pickle.dump({'s1': err_s1_scaler, 's2': err_s2_scaler}, f)

# [SARAN] Optimize weights based on error distribution separation
# Using equal weights initially; can be refined based on evaluation results
combined_val = STAGE1_WEIGHT * err_s1_norm + STAGE2_WEIGHT * err_s2_norm

print(f"\nCombined val error: mean={combined_val.mean():.6f}, std={combined_val.std():.6f}")
print(f"Combined range: [{combined_val.min():.6f}, {combined_val.max():.6f}]")

In [ ]:
# ============================================================
# 11.2 Threshold Strategies
# ============================================================
from scipy.stats import norm

threshold_results = {}

# --- 1. Z-Score ---
mu, sigma = combined_val.mean(), combined_val.std()
for k in K_SIGMA_VALUES:
    t = mu + k * sigma
    fpr = (combined_val > t).mean()
    threshold_results[f'zscore_k{k}'] = {'threshold': float(t), 'fpr': float(fpr), 'method': 'zscore', 'k': k}

# --- 2. Percentile ---
for p in PERCENTILE_VALUES:
    t = np.percentile(combined_val, p)
    fpr = (combined_val > t).mean()
    threshold_results[f'percentile_{p}'] = {'threshold': float(t), 'fpr': float(fpr), 'method': 'percentile', 'p': p}

# --- 3. Gaussian Fit ---
gauss_mu, gauss_sigma = norm.fit(combined_val)
for k in K_SIGMA_VALUES:
    t = gauss_mu + k * gauss_sigma
    fpr = (combined_val > t).mean()
    threshold_results[f'gaussian_k{k}'] = {'threshold': float(t), 'fpr': float(fpr), 'method': 'gaussian', 'k': k}

# --- 4. IQR ---
q1 = np.percentile(combined_val, 25)
q3 = np.percentile(combined_val, 75)
iqr = q3 - q1
for k in [1.5, 2.0, 3.0]:
    t = q3 + k * iqr
    fpr = (combined_val > t).mean()
    threshold_results[f'iqr_k{k}'] = {'threshold': float(t), 'fpr': float(fpr), 'method': 'iqr', 'k': k}

# --- 5. [SARAN] MAD (Median Absolute Deviation) — more robust to outliers ---
median_err = np.median(combined_val)
mad = np.median(np.abs(combined_val - median_err))
for k in K_SIGMA_VALUES:
    t = median_err + k * 1.4826 * mad  # 1.4826 makes MAD consistent with std for normal dist
    fpr = (combined_val > t).mean()
    threshold_results[f'mad_k{k}'] = {'threshold': float(t), 'fpr': float(fpr), 'method': 'mad', 'k': k}

# Print summary
print("=== Threshold Comparison ===")
print(f"{'Method':<20} {'Threshold':>10} {'FPR':>8}")
print("-" * 40)
for name, res in sorted(threshold_results.items(), key=lambda x: x[1]['fpr']):
    print(f"{name:<20} {res['threshold']:>10.6f} {res['fpr']:>8.4f}")

# Select best threshold: Z-Score with smallest k that gives FPR < 0.05
best_threshold_name = None
best_threshold_val = None
for k in sorted(K_SIGMA_VALUES):
    key = f'zscore_k{k}'
    if threshold_results[key]['fpr'] < 0.05:
        best_threshold_name = key
        best_threshold_val = threshold_results[key]['threshold']
        break

if best_threshold_val is None:
    # Fallback to percentile 95
    best_threshold_name = 'percentile_95'
    best_threshold_val = threshold_results['percentile_95']['threshold']
    print(f"\n[INFO] No Z-Score threshold achieved FPR < 0.05, using percentile_95 as fallback")

print(f"\n★ Selected threshold: {best_threshold_name} = {best_threshold_val:.6f} "
      f"(FPR = {threshold_results[best_threshold_name]['fpr']:.4f})")

# Save all thresholds
with open(f"{ARTIFACTS_DIR}/threshold_results.json", 'w') as f:
    json.dump(threshold_results, f, indent=2)
with open(f"{ARTIFACTS_DIR}/best_threshold.json", 'w') as f:
    json.dump({'name': best_threshold_name, 'value': best_threshold_val}, f, indent=2)
print(f"Threshold results saved to {ARTIFACTS_DIR}/")

In [ ]:
# ============================================================
# 11.3 Visualize Error Distribution + Thresholds
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Stage 1 error distribution
axes[0].hist(err_s1_val, bins=100, density=True, alpha=0.7, color='steelblue')
axes[0].set_title('Stage 1 Reconstruction Error (Val)')
axes[0].set_xlabel('MSE')
axes[0].set_ylabel('Density')
axes[0].set_xlim(0, np.percentile(err_s1_val, 99.5))

# Stage 2 error distribution
axes[1].hist(err_s2_val_perflow[~np.isnan(err_s2_val_perflow)], bins=100,
             density=True, alpha=0.7, color='coral')
axes[1].set_title('Stage 2 Reconstruction Error (Val, per-flow)')
axes[1].set_xlabel('MSE')
axes[1].set_ylabel('Density')
axes[1].set_xlim(0, np.percentile(err_s2_val_perflow[~np.isnan(err_s2_val_perflow)], 99.5))

# Combined error + thresholds
axes[2].hist(combined_val, bins=100, density=True, alpha=0.7, color='mediumpurple')
axes[2].axvline(x=best_threshold_val, color='red', linewidth=2, linestyle='--',
                label=f'Best: {best_threshold_name}')
for name, res in threshold_results.items():
    if 'zscore' in name:
        axes[2].axvline(x=res['threshold'], color='gray', linewidth=0.5, alpha=0.5)
axes[2].set_title('Combined Error + Thresholds (Val)')
axes[2].set_xlabel('Weighted Score')
axes[2].set_ylabel('Density')
axes[2].set_xlim(0, np.percentile(combined_val, 99.9))
axes[2].legend()

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/threshold_distributions.png", dpi=150, bbox_inches='tight')
plt.show()

## 12. EVALUATION — CIC-IDS-2017 (Secondary)

Full evaluation pipeline on CIC-IDS-2017 with all attack labels.
Uses saved artifacts (scaler, imputer, feature list) — no re-fitting.

In [ ]:
# ============================================================
# 12.0 Evaluation Helper Functions
# ============================================================
from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                              f1_score, roc_auc_score, average_precision_score,
                              roc_curve, precision_recall_curve)

def preprocess_eval_dataset(df_raw, selected_features, median_values, scaler,
                             col_name_map=None):
    """
    Apply saved preprocessing pipeline to evaluation dataset.
    Returns: X_scaled (np.array), labels (np.array), session_ids, timestamps
    """
    df = df_raw.copy()
    df.columns = df.columns.str.strip()

    # Standardize column names
    rename_map = {}
    for c in df.columns:
        cl = c.lower().strip()
        if 'src' in cl and 'ip' in cl: rename_map[c] = 'Src_IP'
        elif 'dst' in cl and 'ip' in cl: rename_map[c] = 'Dst_IP'
        elif 'source' in cl and 'ip' in cl: rename_map[c] = 'Src_IP'
        elif 'destination' in cl and 'ip' in cl: rename_map[c] = 'Dst_IP'
        elif cl == 'protocol': rename_map[c] = 'Protocol'
        elif 'timestamp' in cl: rename_map[c] = 'Timestamp'
        elif cl == 'label': rename_map[c] = 'Label'
        elif 'flow' in cl and 'id' in cl: rename_map[c] = 'Flow_ID'
        elif ('src' in cl or 'source' in cl) and 'port' in cl: rename_map[c] = 'Src_Port'
        elif ('dst' in cl or 'destination' in cl) and 'port' in cl: rename_map[c] = 'Dst_Port'
    df = df.rename(columns=rename_map)

    # Extract labels
    labels_raw = df['Label'].str.strip() if 'Label' in df.columns else pd.Series(['Unknown'] * len(df))
    y_binary = (~labels_raw.str.upper().isin(['BENIGN'])).astype(int).values
    attack_types = labels_raw.values

    # Session IDs for windowing
    def make_sid(row):
        key = f"{row.get('Src_IP', 'unk')}_{row.get('Dst_IP', 'unk')}_{row.get('Protocol', 'unk')}"
        return hashlib.md5(key.encode()).hexdigest()[:12]
    sids = df.apply(make_sid, axis=1).values

    # Timestamps
    if 'Timestamp' in df.columns:
        try:
            ts = pd.to_datetime(df['Timestamp'], format='mixed', dayfirst=True)
            ts = ts.astype(np.int64) // 10**9
        except Exception:
            ts = np.arange(len(df))
    else:
        ts = np.arange(len(df))
    ts = ts.values if hasattr(ts, 'values') else ts

    # Drop non-feature columns
    drop_cols = ['Label', 'Flow_ID', 'Timestamp', 'Src_IP', 'Dst_IP',
                 'Src_Port', 'Dst_Port', 'session_id', 'ts_parsed']
    df_feat = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

    # Handle Infinity/non-numeric
    for col in df_feat.columns:
        if df_feat[col].dtype == object:
            df_feat[col] = df_feat[col].replace(
                ['Infinity', 'infinity', '-Infinity', '-infinity', 'inf', '-inf'], np.nan
            )
            df_feat[col] = pd.to_numeric(df_feat[col], errors='coerce')
    df_feat = df_feat.replace([np.inf, -np.inf], np.nan)
    df_feat = df_feat.astype(np.float32)

    # Apply median imputation
    for col in df_feat.columns:
        if col in median_values:
            df_feat[col] = df_feat[col].fillna(float(median_values[col]))
        else:
            df_feat[col] = df_feat[col].fillna(0.0)

    # Select features (handle missing features)
    X_df = pd.DataFrame()
    for sf in selected_features:
        if sf in df_feat.columns:
            X_df[sf] = df_feat[sf]
        else:
            X_df[sf] = 0.0
            print(f"  [WARN] Feature '{sf}' not found in dataset, using 0")

    # Scale + clip
    X_scaled = scaler.transform(X_df.values).astype(np.float32)
    X_scaled = np.clip(X_scaled, -5, 5)

    return X_scaled, y_binary, attack_types, sids, ts


def compute_eval_scores(X_scaled, sids, ts, model_s1, encoder_s1, model_s2,
                         err_s1_scaler, err_s2_scaler, window_size, min_flows):
    """Compute combined anomaly scores for evaluation data."""
    # Stage 1: per-flow error
    X_s1 = X_scaled[..., np.newaxis]
    X_s1_pred = model_s1.predict(X_s1, batch_size=S1_BATCH_SIZE, verbose=0)
    err_s1 = np.mean((X_scaled - X_s1_pred.reshape(X_scaled.shape)) ** 2, axis=1)

    # Stage 1: latent extraction
    latent = encoder_s1.predict(X_s1, batch_size=S1_BATCH_SIZE, verbose=0)

    # Windowing in latent space
    windows, masks, flow_indices = create_session_windows(
        latent, sids, ts, window_size, min_flows
    )

    # Stage 2: per-window error
    if len(windows) > 0:
        err_s2_windows = compute_recon_error(model_s2, windows)

        # Map back to per-flow
        err_s2_perflow = np.full(len(X_scaled), np.nan, dtype=np.float32)
        for w_idx in range(len(flow_indices)):
            fi = flow_indices[w_idx]
            valid = fi[fi >= 0]
            err_s2_perflow[valid] = err_s2_windows[w_idx]

        not_windowed = np.isnan(err_s2_perflow)
        if not_windowed.any():
            err_s2_perflow[not_windowed] = np.nanmedian(err_s2_perflow)
    else:
        err_s2_perflow = np.zeros(len(X_scaled), dtype=np.float32)

    # Normalize using saved scalers (transform only, no fit)
    err_s1_norm = err_s1_scaler.transform(err_s1.reshape(-1, 1)).ravel()
    err_s2_norm = err_s2_scaler.transform(err_s2_perflow.reshape(-1, 1)).ravel()

    # Clip to [0, 1] to handle out-of-range values
    err_s1_norm = np.clip(err_s1_norm, 0, 1)
    err_s2_norm = np.clip(err_s2_norm, 0, 1)

    combined = STAGE1_WEIGHT * err_s1_norm + STAGE2_WEIGHT * err_s2_norm
    return combined, err_s1, err_s2_perflow


def evaluate_with_thresholds(y_true, y_score, threshold_results, dataset_name):
    """Evaluate all threshold strategies and return metrics."""
    all_metrics = {}

    for t_name, t_info in threshold_results.items():
        t_val = t_info['threshold']
        y_pred = (y_score > t_val).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        fpr_val = fp / (fp + tn) if (fp + tn) > 0 else 0

        all_metrics[t_name] = {
            'threshold': t_val,
            'precision': float(precision_score(y_true, y_pred, zero_division=0)),
            'recall': float(recall_score(y_true, y_pred, zero_division=0)),
            'f1': float(f1_score(y_true, y_pred, zero_division=0)),
            'fpr': float(fpr_val),
            'tp': int(tp), 'fp': int(fp), 'tn': int(tn), 'fn': int(fn),
        }

    # ROC-AUC and PR-AUC are threshold-independent
    try:
        roc_auc = float(roc_auc_score(y_true, y_score))
    except Exception:
        roc_auc = 0.0
    try:
        pr_auc = float(average_precision_score(y_true, y_score))
    except Exception:
        pr_auc = 0.0

    for k in all_metrics:
        all_metrics[k]['roc_auc'] = roc_auc
        all_metrics[k]['pr_auc'] = pr_auc

    return all_metrics, roc_auc, pr_auc

print("Evaluation helper functions defined.")

In [ ]:
# ============================================================
# 12.1 Evaluate CIC-IDS-2017 (All Labels)
# ============================================================
print("=== Evaluating CIC-IDS-2017 (All Labels) ===")

# Load full CIC-2017 data
df_cic_all = pd.read_parquet(f"{PREPROCESSED_DIR}/cic2017_all_raw.parquet")
print(f"CIC-2017 all labels: {len(df_cic_all):,} rows")
print(f"Label distribution:\n{df_cic_all['Label'].value_counts() if 'Label' in df_cic_all.columns else 'N/A'}")

# Load error scalers
with open(f"{ARTIFACTS_DIR}/error_scalers.pkl", 'rb') as f:
    err_scalers = pickle.load(f)

X_cic, y_cic, attacks_cic, sids_cic, ts_cic = preprocess_eval_dataset(
    df_cic_all, selected_features, median_values, scaler
)
print(f"Preprocessed: {X_cic.shape}, attacks: {len(np.unique(attacks_cic))} types")
print(f"Benign: {(y_cic == 0).sum():,}, Attack: {(y_cic == 1).sum():,}")

scores_cic, err_s1_cic, err_s2_cic = compute_eval_scores(
    X_cic, sids_cic, ts_cic, model_s1, encoder_s1, model_s2,
    err_scalers['s1'], err_scalers['s2'], WINDOW_SIZE, MIN_FLOWS_PER_SESSION
)
print(f"Scores computed: mean={scores_cic.mean():.4f}, std={scores_cic.std():.4f}")

metrics_cic, roc_auc_cic, pr_auc_cic = evaluate_with_thresholds(
    y_cic, scores_cic, threshold_results, 'CIC-2017'
)

# [PERINGATAN] Check for inverted scores
if roc_auc_cic < 0.5:
    print(f"[PERINGATAN] ROC-AUC = {roc_auc_cic:.4f} < 0.5!")
    print("  Scores may be inverted — attacks have LOWER error than benign.")
    print("  This could indicate the model reconstructs attacks better than benign.")
    print("  Attempting score inversion...")
    scores_cic_inv = 1 - scores_cic
    roc_auc_inv = roc_auc_score(y_cic, scores_cic_inv)
    if roc_auc_inv > roc_auc_cic:
        print(f"  Inverted ROC-AUC = {roc_auc_inv:.4f} — using inverted scores")
        scores_cic = scores_cic_inv
        roc_auc_cic = roc_auc_inv
        metrics_cic, roc_auc_cic, pr_auc_cic = evaluate_with_thresholds(
            y_cic, scores_cic, threshold_results, 'CIC-2017'
        )

print(f"\n=== CIC-2017 Results (best threshold: {best_threshold_name}) ===")
best_m = metrics_cic[best_threshold_name]
print(f"ROC-AUC   : {roc_auc_cic:.4f}")
print(f"PR-AUC    : {pr_auc_cic:.4f}")
print(f"Precision : {best_m['precision']:.4f}")
print(f"Recall    : {best_m['recall']:.4f}")
print(f"F1-Score  : {best_m['f1']:.4f}")
print(f"FPR       : {best_m['fpr']:.4f}")

del df_cic_all
gc.collect()

In [ ]:
# ============================================================
# 12.2 CIC-2017 Per-Attack Detection Rate
# ============================================================
best_t = threshold_results[best_threshold_name]['threshold']
y_pred_cic = (scores_cic > best_t).astype(int)

print("=== CIC-2017: Per-Attack Detection Rate ===")
attack_df_cic = pd.DataFrame({'attack': attacks_cic, 'y_true': y_cic, 'y_pred': y_pred_cic})

det_rates_cic = {}
for atype, grp in attack_df_cic.groupby('attack'):
    if atype.strip().upper() == 'BENIGN':
        continue
    dr = grp['y_pred'].mean()
    det_rates_cic[atype] = {'detection_rate': dr, 'count': len(grp)}
    status = '✓' if dr > 0.7 else ('⚠' if dr > 0.3 else '✗')
    print(f"  {status} {atype:<30} DR={dr:.4f}  (n={len(grp):,})")

# Benign FPR
benign_grp = attack_df_cic[attack_df_cic['attack'].str.strip().str.upper() == 'BENIGN']
benign_fpr = benign_grp['y_pred'].mean()
print(f"\n  Benign FPR: {benign_fpr:.4f}")

In [ ]:
# ============================================================
# 12.3 CIC-2017 Visualizations
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Error distribution per class (violin)
unique_attacks = [a for a in np.unique(attacks_cic) if a.strip().upper() != 'BENIGN']
plot_data = [scores_cic[attacks_cic == 'BENIGN']] if 'BENIGN' in attacks_cic else []
plot_labels = ['BENIGN'] if 'BENIGN' in attacks_cic else []

# Use canonical benign label
benign_mask_cic = pd.Series(attacks_cic).str.strip().str.upper() == 'BENIGN'
plot_data = [scores_cic[benign_mask_cic.values]]
plot_labels = ['BENIGN']

for a in sorted(unique_attacks)[:8]:
    mask = attacks_cic == a
    if mask.sum() > 10:
        plot_data.append(scores_cic[mask])
        plot_labels.append(a[:20])

bp = axes[0, 0].boxplot(plot_data, labels=plot_labels, vert=True, patch_artist=True)
for i, box in enumerate(bp['boxes']):
    box.set_facecolor('lightblue' if i == 0 else 'salmon')
axes[0, 0].axhline(y=best_t, color='red', linestyle='--', label='Threshold')
axes[0, 0].set_title('CIC-2017: Error Distribution by Class')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].set_ylabel('Anomaly Score')
axes[0, 0].legend()

# 2. ROC Curve
fpr_curve, tpr_curve, _ = roc_curve(y_cic, scores_cic)
axes[0, 1].plot(fpr_curve, tpr_curve, 'b-', label=f'CIC-2017 (AUC={roc_auc_cic:.4f})')
axes[0, 1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('ROC Curve — CIC-2017')
axes[0, 1].legend()

# 3. PR Curve
prec_curve, rec_curve, _ = precision_recall_curve(y_cic, scores_cic)
axes[1, 0].plot(rec_curve, prec_curve, 'b-', label=f'CIC-2017 (AP={pr_auc_cic:.4f})')
axes[1, 0].set_xlabel('Recall')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_title('Precision-Recall Curve — CIC-2017')
axes[1, 0].legend()

# 4. Confusion Matrix
cm = confusion_matrix(y_cic, y_pred_cic, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 1],
            xticklabels=['Benign', 'Attack'], yticklabels=['Benign', 'Attack'])
axes[1, 1].set_xlabel('Predicted')
axes[1, 1].set_ylabel('Actual')
axes[1, 1].set_title(f'CIC-2017 Confusion Matrix ({best_threshold_name})')

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/cic2017_evaluation.png", dpi=150, bbox_inches='tight')
plt.show()

# Detection rate bar chart
fig2, ax2 = plt.subplots(figsize=(12, 5))
dr_names = list(det_rates_cic.keys())
dr_vals = [det_rates_cic[k]['detection_rate'] for k in dr_names]
colors_dr = ['#27ae60' if v > 0.7 else '#f39c12' if v > 0.3 else '#e74c3c' for v in dr_vals]
ax2.barh(dr_names, dr_vals, color=colors_dr)
ax2.axvline(x=0.7, color='green', linestyle='--', alpha=0.5, label='Good (>0.7)')
ax2.axvline(x=0.3, color='red', linestyle='--', alpha=0.5, label='Poor (<0.3)')
ax2.set_xlabel('Detection Rate')
ax2.set_title('CIC-2017: Detection Rate per Attack Category')
ax2.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/cic2017_detection_rates.png", dpi=150, bbox_inches='tight')
plt.show()

# Save CIC results
pd.DataFrame(metrics_cic).T.to_csv(f"{RESULTS_DIR}/cic2017_metrics.csv")
print("CIC-2017 evaluation complete. Results saved.")

## 13. EVALUATION — CSE-CIC-IDS-2018 (Primary)

**This is the primary evaluation target** — model has never seen this dataset.
Chunk-based loading to handle large file sizes.

In [ ]:
# ============================================================
# 13.1 Load & Evaluate CSE-CIC-IDS-2018
# ============================================================
print("=== Evaluating CSE-CIC-IDS-2018 (All Labels) ===")

cse_csv_files = sorted([
    os.path.join(CSE2018_DIR, f)
    for f in os.listdir(CSE2018_DIR) if f.endswith('.csv')
])

all_X_cse, all_y_cse, all_attacks_cse, all_sids_cse, all_ts_cse = [], [], [], [], []

for fpath in cse_csv_files:
    fname = os.path.basename(fpath)
    print(f"  Processing {fname}...", end=" ")

    try:
        chunks_data = []
        for chunk in pd.read_csv(fpath, chunksize=CHUNK_SIZE, low_memory=False, encoding='utf-8'):
            chunks_data.append(chunk)
        df_cse_file = pd.concat(chunks_data, ignore_index=True)

        X_cse_f, y_cse_f, atk_cse_f, sid_cse_f, ts_cse_f = preprocess_eval_dataset(
            df_cse_file, selected_features, median_values, scaler
        )

        all_X_cse.append(X_cse_f)
        all_y_cse.append(y_cse_f)
        all_attacks_cse.append(atk_cse_f)
        all_sids_cse.append(sid_cse_f)
        all_ts_cse.append(ts_cse_f)
        print(f"{len(df_cse_file):,} rows (benign: {(y_cse_f==0).sum():,}, attack: {(y_cse_f==1).sum():,})")

        del df_cse_file, chunks_data
        gc.collect()
    except Exception as e:
        print(f"ERROR: {e}")
        continue

X_cse = np.vstack(all_X_cse)
y_cse = np.concatenate(all_y_cse)
attacks_cse = np.concatenate(all_attacks_cse)
sids_cse = np.concatenate(all_sids_cse)
ts_cse = np.concatenate(all_ts_cse)

print(f"\nCSE-2018 total: {len(X_cse):,} rows")
print(f"Benign: {(y_cse == 0).sum():,}, Attack: {(y_cse == 1).sum():,}")
print(f"Attack types: {len(np.unique(attacks_cse[y_cse == 1]))}:")
for atype in np.unique(attacks_cse):
    n = (attacks_cse == atype).sum()
    print(f"  {atype}: {n:,}")

del all_X_cse, all_y_cse, all_attacks_cse, all_sids_cse, all_ts_cse
gc.collect()

In [ ]:
# ============================================================
# 13.2 Compute Scores & Metrics for CSE-2018
# ============================================================
scores_cse, err_s1_cse, err_s2_cse = compute_eval_scores(
    X_cse, sids_cse, ts_cse, model_s1, encoder_s1, model_s2,
    err_scalers['s1'], err_scalers['s2'], WINDOW_SIZE, MIN_FLOWS_PER_SESSION
)
print(f"Scores computed: mean={scores_cse.mean():.4f}, std={scores_cse.std():.4f}")

metrics_cse, roc_auc_cse, pr_auc_cse = evaluate_with_thresholds(
    y_cse, scores_cse, threshold_results, 'CSE-2018'
)

# [PERINGATAN] Critical check — same issue as previous implementation?
if roc_auc_cse < 0.5:
    print(f"\n[PERINGATAN] ROC-AUC CSE = {roc_auc_cse:.4f} < 0.5!")
    print("  This was the critical failure mode of the previous implementation (0.3633).")
    print("  Investigating score inversion...")
    scores_cse_inv = 1 - scores_cse
    roc_auc_inv = roc_auc_score(y_cse, scores_cse_inv)
    if roc_auc_inv > roc_auc_cse:
        print(f"  Inverted ROC-AUC = {roc_auc_inv:.4f}")
        print("  Root cause: model reconstructs CSE attacks BETTER than CSE benign")
        print("  This is a domain shift problem, not a score direction problem.")
        # Keep original scores for honest reporting
    print("  Reporting original (non-inverted) scores for honest evaluation.")

best_m_cse = metrics_cse[best_threshold_name]
print(f"\n=== CSE-2018 Results (best threshold: {best_threshold_name}) ===")
print(f"ROC-AUC   : {roc_auc_cse:.4f}")
print(f"PR-AUC    : {pr_auc_cse:.4f}")
print(f"Precision : {best_m_cse['precision']:.4f}")
print(f"Recall    : {best_m_cse['recall']:.4f}")
print(f"F1-Score  : {best_m_cse['f1']:.4f}")
print(f"FPR       : {best_m_cse['fpr']:.4f}")

In [ ]:
# ============================================================
# 13.3 CSE-2018 Per-Attack Detection Rate
# ============================================================
y_pred_cse = (scores_cse > best_t).astype(int)

print("=== CSE-2018: Per-Attack Detection Rate ===")
attack_df_cse = pd.DataFrame({'attack': attacks_cse, 'y_true': y_cse, 'y_pred': y_pred_cse})

det_rates_cse = {}
for atype, grp in attack_df_cse.groupby('attack'):
    if atype.strip().upper() == 'BENIGN':
        continue
    dr = grp['y_pred'].mean()
    det_rates_cse[atype] = {'detection_rate': dr, 'count': len(grp)}
    status = '✓' if dr > 0.7 else ('⚠' if dr > 0.3 else '✗')
    print(f"  {status} {atype:<30} DR={dr:.4f}  (n={len(grp):,})")

# Benign FPR
benign_grp_cse = attack_df_cse[attack_df_cse['attack'].str.strip().str.upper() == 'BENIGN']
benign_fpr_cse = benign_grp_cse['y_pred'].mean() if len(benign_grp_cse) > 0 else float('nan')
print(f"\n  Benign FPR: {benign_fpr_cse:.4f}")

In [ ]:
# ============================================================
# 13.4 CSE-2018 Visualizations
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Error distribution by class
benign_mask_cse = pd.Series(attacks_cse).str.strip().str.upper() == 'BENIGN'
unique_attacks_cse = [a for a in np.unique(attacks_cse)
                       if a.strip().upper() != 'BENIGN']

plot_data_cse = [scores_cse[benign_mask_cse.values]]
plot_labels_cse = ['BENIGN']
for a in sorted(unique_attacks_cse)[:8]:
    mask = attacks_cse == a
    if mask.sum() > 10:
        plot_data_cse.append(scores_cse[mask])
        plot_labels_cse.append(a[:20])

bp = axes[0, 0].boxplot(plot_data_cse, labels=plot_labels_cse, vert=True, patch_artist=True)
for i, box in enumerate(bp['boxes']):
    box.set_facecolor('lightblue' if i == 0 else 'salmon')
axes[0, 0].axhline(y=best_t, color='red', linestyle='--', label='Threshold')
axes[0, 0].set_title('CSE-2018: Error Distribution by Class')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].set_ylabel('Anomaly Score')
axes[0, 0].legend()

# 2. ROC Curve (both datasets)
fpr_cse_curve, tpr_cse_curve, _ = roc_curve(y_cse, scores_cse)
fpr_cic_curve, tpr_cic_curve, _ = roc_curve(y_cic, scores_cic)
axes[0, 1].plot(fpr_cic_curve, tpr_cic_curve, 'b-', label=f'CIC-2017 (AUC={roc_auc_cic:.4f})', alpha=0.7)
axes[0, 1].plot(fpr_cse_curve, tpr_cse_curve, 'r-', label=f'CSE-2018 (AUC={roc_auc_cse:.4f})', alpha=0.7)
axes[0, 1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('ROC Curves — CIC vs CSE')
axes[0, 1].legend()

# 3. PR Curve (both datasets)
prec_cse, rec_cse, _ = precision_recall_curve(y_cse, scores_cse)
prec_cic_c, rec_cic_c, _ = precision_recall_curve(y_cic, scores_cic)
axes[1, 0].plot(rec_cic_c, prec_cic_c, 'b-', label=f'CIC-2017 (AP={pr_auc_cic:.4f})', alpha=0.7)
axes[1, 0].plot(rec_cse, prec_cse, 'r-', label=f'CSE-2018 (AP={pr_auc_cse:.4f})', alpha=0.7)
axes[1, 0].set_xlabel('Recall')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_title('Precision-Recall Curves — CIC vs CSE')
axes[1, 0].legend()

# 4. Confusion Matrix CSE
cm_cse = confusion_matrix(y_cse, y_pred_cse, labels=[0, 1])
sns.heatmap(cm_cse, annot=True, fmt='d', cmap='Reds', ax=axes[1, 1],
            xticklabels=['Benign', 'Attack'], yticklabels=['Benign', 'Attack'])
axes[1, 1].set_xlabel('Predicted')
axes[1, 1].set_ylabel('Actual')
axes[1, 1].set_title(f'CSE-2018 Confusion Matrix ({best_threshold_name})')

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/cse2018_evaluation.png", dpi=150, bbox_inches='tight')
plt.show()

# Detection rate bar chart
fig2, ax2 = plt.subplots(figsize=(12, 5))
dr_names_cse = list(det_rates_cse.keys())
dr_vals_cse = [det_rates_cse[k]['detection_rate'] for k in dr_names_cse]
colors_dr_cse = ['#27ae60' if v > 0.7 else '#f39c12' if v > 0.3 else '#e74c3c' for v in dr_vals_cse]
ax2.barh(dr_names_cse, dr_vals_cse, color=colors_dr_cse)
ax2.axvline(x=0.7, color='green', linestyle='--', alpha=0.5, label='Good (>0.7)')
ax2.axvline(x=0.3, color='red', linestyle='--', alpha=0.5, label='Poor (<0.3)')
ax2.set_xlabel('Detection Rate')
ax2.set_title('CSE-2018: Detection Rate per Attack Category')
ax2.legend()
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/cse2018_detection_rates.png", dpi=150, bbox_inches='tight')
plt.show()

# Save CSE results
pd.DataFrame(metrics_cse).T.to_csv(f"{RESULTS_DIR}/cse2018_metrics.csv")
print("CSE-2018 evaluation complete. Results saved.")

## 14. GENERALIZATION ANALYSIS

Quantitative comparison of model performance across datasets.
Determines whether the model generalizes, partially generalizes, or is CIC-specific.

In [ ]:
# ============================================================
# 14.1 Side-by-Side Comparison
# ============================================================
print("=" * 70)
print("GENERALIZATION ANALYSIS: CIC-2017 vs CSE-2018")
print("=" * 70)

best_cic = metrics_cic[best_threshold_name]
best_cse = metrics_cse[best_threshold_name]

comparison_rows = []
targets = {
    'ROC-AUC':   (roc_auc_cic, roc_auc_cse, 0.85, 0.75),
    'PR-AUC':    (pr_auc_cic, pr_auc_cse, 0.70, 0.60),
    'F1-Score':  (best_cic['f1'], best_cse['f1'], 0.65, 0.50),
    'Recall':    (best_cic['recall'], best_cse['recall'], 0.65, 0.50),
    'Precision': (best_cic['precision'], best_cse['precision'], 0.65, 0.50),
    'FPR':       (best_cic['fpr'], best_cse['fpr'], 0.05, 0.10),
}

print(f"\n{'Metric':<12} {'CIC-2017':>10} {'CSE-2018':>10} {'Gap':>8} {'Target CIC':>11} {'Target CSE':>11} {'Status':>8}")
print("-" * 75)

statuses = []
for metric, (cic_val, cse_val, target_cic, target_cse) in targets.items():
    gap = cse_val - cic_val
    if metric == 'FPR':
        cic_ok = cic_val <= target_cic
        cse_ok = cse_val <= target_cse
    else:
        cic_ok = cic_val >= target_cic
        cse_ok = cse_val >= target_cse

    status = '✓' if cse_ok else ('⚠' if abs(cse_val - target_cse) < 0.1 else '✗')
    statuses.append(status)

    print(f"{metric:<12} {cic_val:>10.4f} {cse_val:>10.4f} {gap:>+8.4f} "
          f"{'<' if metric == 'FPR' else '>'}{target_cic:>10.2f} "
          f"{'<' if metric == 'FPR' else '>'}{target_cse:>10.2f} {status:>8}")

    comparison_rows.append({
        'metric': metric, 'cic_2017': cic_val, 'cse_2018': cse_val,
        'gap': gap, 'target_cic': target_cic, 'target_cse': target_cse,
        'status': status
    })

pd.DataFrame(comparison_rows).to_csv(f"{RESULTS_DIR}/generalization_comparison.csv", index=False)

In [ ]:
# ============================================================
# 14.2 Attack-Level Analysis
# ============================================================
print("\n=== Attack-Level Analysis ===")

# High detection attacks
high_det_cic = {k: v for k, v in det_rates_cic.items() if v['detection_rate'] > 0.7}
low_det_cic = {k: v for k, v in det_rates_cic.items() if v['detection_rate'] < 0.3}
high_det_cse = {k: v for k, v in det_rates_cse.items() if v['detection_rate'] > 0.7}
low_det_cse = {k: v for k, v in det_rates_cse.items() if v['detection_rate'] < 0.3}

print(f"\nCIC-2017 — Successfully detected (DR > 0.7): {len(high_det_cic)} attack types")
for k, v in sorted(high_det_cic.items(), key=lambda x: -x[1]['detection_rate']):
    print(f"  ✓ {k:<30} DR={v['detection_rate']:.4f}")

print(f"\nCIC-2017 — Failed to detect (DR < 0.3): {len(low_det_cic)} attack types")
for k, v in sorted(low_det_cic.items(), key=lambda x: x[1]['detection_rate']):
    print(f"  ✗ {k:<30} DR={v['detection_rate']:.4f}")

print(f"\nCSE-2018 — Successfully detected (DR > 0.7): {len(high_det_cse)} attack types")
for k, v in sorted(high_det_cse.items(), key=lambda x: -x[1]['detection_rate']):
    print(f"  ✓ {k:<30} DR={v['detection_rate']:.4f}")

print(f"\nCSE-2018 — Failed to detect (DR < 0.3): {len(low_det_cse)} attack types")
for k, v in sorted(low_det_cse.items(), key=lambda x: x[1]['detection_rate']):
    print(f"  ✗ {k:<30} DR={v['detection_rate']:.4f}")

# Correlate with domain shift
if os.path.exists(f"{RESULTS_DIR}/domain_shift_analysis.csv"):
    ks_df = pd.read_csv(f"{RESULTS_DIR}/domain_shift_analysis.csv")
    high_shift_feats = ks_df[ks_df['shift'] == 'high']['feature'].tolist()
    print(f"\n[ANALISIS] Features with high domain shift: {len(high_shift_feats)}")
    print(f"  These features may explain poor CSE detection for some attack types.")
    print(f"  High-shift features: {high_shift_feats[:5]}{'...' if len(high_shift_feats) > 5 else ''}")

In [ ]:
# ============================================================
# 14.3 Generalization Verdict
# ============================================================
n_pass = statuses.count('✓')
n_warn = statuses.count('⚠')
n_fail = statuses.count('✗')

if n_pass >= 4 and roc_auc_cse > 0.70:
    verdict = "GENERAL"
    verdict_desc = ("Model successfully generalizes to CSE-2018. "
                    "Performance on unseen dataset is close to training dataset.")
elif roc_auc_cse > 0.55 and n_pass + n_warn >= 3:
    verdict = "PARTIAL"
    verdict_desc = ("Model partially generalizes. Some attack types are detected, "
                    "but significant performance gap exists between CIC and CSE.")
else:
    verdict = "CIC-SPECIFIC"
    verdict_desc = ("Model is overfit to CIC-2017 data distribution. "
                    "CSE-2018 performance is far below targets. "
                    "Domain adaptation techniques required for Phase 3.")

print(f"\n{'='*70}")
print(f"GENERALIZATION VERDICT: {verdict}")
print(f"{'='*70}")
print(f"\n{verdict_desc}")
print(f"\nMetric summary: {n_pass} pass, {n_warn} warning, {n_fail} fail")
print(f"ROC-AUC gap: CIC={roc_auc_cic:.4f} → CSE={roc_auc_cse:.4f} (Δ={roc_auc_cse - roc_auc_cic:+.4f})")

if roc_auc_cse < 0.5:
    print("\n[PERINGATAN KRITIS] ROC-AUC CSE < 0.5 — JANGAN lanjut ke Fase 3!")
    print("  Model perlu diagnosis fundamental sebelum refinement.")
    print("  Kemungkinan root cause:")
    print("  1. Domain shift terlalu besar pada fitur-fitur kunci")
    print("  2. Preprocessing tidak konsisten antara CIC dan CSE")
    print("  3. Model belajar noise CIC-specific, bukan anomaly patterns universal")

## 15. LAPORAN AKHIR

In [ ]:
# ============================================================
# 15. Generate Final Report
# ============================================================
report_date = datetime.now().strftime('%Y-%m-%d %H:%M')

# Gather training info
try:
    s1_best_epoch = history_s1.history['val_loss'].index(min(history_s1.history['val_loss'])) + 1
    s1_best_val = min(history_s1.history['val_loss'])
except Exception:
    s1_best_epoch = 'N/A'
    s1_best_val = 'N/A'

try:
    s2_best_epoch = history_s2.history['val_loss'].index(min(history_s2.history['val_loss'])) + 1
    s2_best_val = min(history_s2.history['val_loss'])
except Exception:
    s2_best_epoch = 'N/A'
    s2_best_val = 'N/A'

# Load domain shift info
try:
    ks_df = pd.read_csv(f"{RESULTS_DIR}/domain_shift_analysis.csv")
    n_ks_low = (ks_df['shift'] == 'low').sum()
    n_ks_med = (ks_df['shift'] == 'medium').sum()
    n_ks_high = (ks_df['shift'] == 'high').sum()
except Exception:
    n_ks_low = n_ks_med = n_ks_high = 'N/A'

# Session stats
try:
    session_lens = pd.Series(train_session_ids).value_counts()
    median_sess = session_lens.median()
    pct_single_sess = (session_lens == 1).mean() * 100
    pct_padded = masks_train.any(axis=1).mean() * 100
except Exception:
    median_sess = pct_single_sess = pct_padded = 'N/A'

# Threshold comparison table
threshold_table = "| Method | Threshold | FPR | Precision | Recall | F1 |\n"
threshold_table += "|--------|-----------|-----|-----------|--------|-----|\n"
for t_name in sorted(threshold_results.keys()):
    t = threshold_results[t_name]
    m_cic = metrics_cic.get(t_name, {})
    threshold_table += (f"| {t_name} | {t['threshold']:.6f} | {t['fpr']:.4f} | "
                        f"{m_cic.get('precision', 0):.4f} | {m_cic.get('recall', 0):.4f} | "
                        f"{m_cic.get('f1', 0):.4f} |\n")

report = f"""# Laporan Akhir Fase 2 — MSCNN-BiLSTM AE
Tanggal: {report_date}

## 1. Arsitektur Final
- **Stage 1 MSCNN-AE**: Multi-Scale Conv1D (kernels 3,5,7) → GAP → Dense({S1_LATENT_DIM})
- **Stage 2 BiLSTM-AE**: Bidirectional LSTM (32→16) → Dense({S2_LATENT_DIM}) → RepeatVector → BiLSTM (16→32)
- Compression ratio Stage 1: {n_features} / {S1_LATENT_DIM} = {n_features/S1_LATENT_DIM:.1f}x
- Compression ratio Stage 2: ({WINDOW_SIZE} × {S1_LATENT_DIM}) / {S2_LATENT_DIM} = {(WINDOW_SIZE * S1_LATENT_DIM)/S2_LATENT_DIM:.1f}x
- Total params Stage 1: {model_s1.count_params():,}
- Total params Stage 2: {model_s2.count_params():,}

## 2. Feature Selection Summary
- Fitur awal: {len(df_numeric.columns) if 'df_numeric' in dir() else 'N/A'}
- Setelah NZV filter: {len(df_numeric.columns) - len(low_var_features) if 'low_var_features' in dir() else 'N/A'}
- Setelah corr filter: {n_features}
- Fitur final yang digunakan: {n_features}

## 3. Domain Shift Summary
- Fitur low shift (KS < 0.1): {n_ks_low}
- Fitur medium shift (0.1 ≤ KS < 0.3): {n_ks_med}
- Fitur high shift (KS ≥ 0.3): {n_ks_high}
- Rekomendasi: Keep all features, RobustScaler + clipping for mitigation

## 4. Session Windowing Statistics
- Window size: {WINDOW_SIZE}
- Median session length: {median_sess} flows
- % Sessions with 1 flow: {pct_single_sess}%
- % Windows padded: {pct_padded}%
- Mode: {'session-based' if WINDOW_SIZE > 1 else 'per-flow (fallback)'}

## 5. Training Summary
- Stage 1: stopped at epoch {s1_best_epoch}, best val_loss = {s1_best_val}
- Stage 2: stopped at epoch {s2_best_epoch}, best val_loss = {s2_best_val}

## 6. Threshold Comparison (CIC-2017 metrics)
{threshold_table}
Selected: **{best_threshold_name}** (threshold = {best_threshold_val:.6f})

## 7. Metrik Akhir

| Metric | CIC-2017 | CSE-2018 | Gap |
|--------|----------|----------|-----|
| ROC-AUC | {roc_auc_cic:.4f} | {roc_auc_cse:.4f} | {roc_auc_cse - roc_auc_cic:+.4f} |
| PR-AUC | {pr_auc_cic:.4f} | {pr_auc_cse:.4f} | {pr_auc_cse - pr_auc_cic:+.4f} |
| F1-Score | {best_cic['f1']:.4f} | {best_cse['f1']:.4f} | {best_cse['f1'] - best_cic['f1']:+.4f} |
| Recall | {best_cic['recall']:.4f} | {best_cse['recall']:.4f} | {best_cse['recall'] - best_cic['recall']:+.4f} |
| FPR | {best_cic['fpr']:.4f} | {best_cse['fpr']:.4f} | {best_cse['fpr'] - best_cic['fpr']:+.4f} |

## 8. Verdict Generalization
**{verdict}** — {verdict_desc}

## 9. Top 3 Temuan Penting
1. Compression ratio Stage 1 = {n_features/S1_LATENT_DIM:.1f}x forces model to learn compressed representations instead of memorizing
2. Session-based windowing in latent space (W={WINDOW_SIZE}) captures temporal patterns without raw-feature noise
3. ROC-AUC gap CIC→CSE = {roc_auc_cse - roc_auc_cic:+.4f} {'indicates successful generalization' if abs(roc_auc_cse - roc_auc_cic) < 0.15 else 'indicates domain shift challenge'}

## 10. Rekomendasi Fase 3
1. **Isolation Forest ensemble**: Use Stage 1 + Stage 2 reconstruction errors as features for IF
2. **Adaptive weighting**: Optimize STAGE1_WEIGHT/STAGE2_WEIGHT based on observed error distributions
3. **Domain adaptation**: If CSE gap persists, consider feature-wise normalization or adversarial alignment
"""

with open(f"{RESULTS_DIR}/fase2_report.md", 'w') as f:
    f.write(report)

print(report)
print(f"\n\nReport saved to: {RESULTS_DIR}/fase2_report.md")
print("\n" + "=" * 70)
print("FASE 2 COMPLETE")
print("=" * 70)